> **Notebook d'ÉVALUATION — entraînement déjà terminé (50/50 époques, 8,69 h).**
>
> **Avant de lancer :** `Add Data` → ton Dataset de checkpoints (monté sous `/kaggle/input`, remonté à chaque session).
>
> **Ordre :** tout depuis le haut (installation → Config → `.pt` → index → splits → YAML → clonage MOTFM → **patches**), puis restauration du checkpoint, puis §10 → §13. La cellule d'entraînement est **désactivée** (`RUN_TRAINING=False`).
>
> **Corrections :**
> - `LAZY v2` : respecte `split_val` (l'inférence tournait sinon sur *val* au lieu de *test*) et renseigne `val_data` (sinon `inferer.py` lève *Failed to initialize validation data*).
> - §10 : `test_list` reconstruit depuis les `.pt` — `motfm_data.pkl` n'existe plus (supprimé avec le chargement paresseux). C'était le `FileNotFoundError`.
> - §10–§12 : `data_range` **2.0 → 1.0** (images en `[0,1]`, pas `[-1,1]`) — sinon SSIM surestimé.
> - 3 cellules redondantes/cassées supprimées.
>
> **Point de contrôle :** la §10 doit logger `[LAZY] split 'test' -> 2573 echantillons`. Si `'val'` ou 2 719 apparaît, l'alignement prédictions ↔ vérités terrain est rompu.


> **Version Kaggle finale — la meilleure config pour 2×T4 / 32 Go RAM / disque limité :**
>
> **Recommandé (persistant, zéro disque) :** prépare les `.pt` **une fois** dans un *Kaggle Dataset*, attache-le, et mets son chemin dans `CFG.kaggle_input_dir`. Le chargement paresseux lit alors directement `/kaggle/input` (lecture seule) : aucun téléchargement, aucun octet de disque consommé, et le DDP 2×T4 fonctionne.
>
> **Étapes (une seule fois) :** lance le notebook sans `kaggle_input_dir` → il télécharge l'archive HF de façon **disque-safe** (fichier local, ~45 Go de pic, pas de bug *Cannot seek*) et extrait les `.pt`. Passe `MAKE_KAGGLE_DATASET=True` dans la cellule dédiée pour publier un Dataset. Puis *Add Input* + renseigne `CFG.kaggle_input_dir`.
>
> Ensuite, à chaque session : **0 téléchargement**, données en lecture seule, checkpoints sur `/kaggle/working` (persistants, reprise `ckpt_path=last`). Conserve tout le reste : Dataset paresseux, DDP, AdamW+cosine+warmup, EMA, `torch.compile`, TF32, modèle réduit 47,2 M.
>
> ⚠️ Environnement Kaggle = **NumPy 2.x** : ne jamais faire `pip install "numpy<2"` ni réinstaller `datasets` (déjà présent). La cellule d'installation gèle NumPy pour éviter la rupture d'ABI.


> **Solution B — Kaggle 2×T4, données complètes, chargement paresseux :**
>
> - **Dataset paresseux** (`LazyPtDataset`) : lit chaque `.pt` à la demande sur le disque, **aucune copie du dataset en RAM** → plus de pickle géant, plus d'OOM. Utilise **100 %** des données.
> - **DDP sur les 2 GPU T4** de nouveau possible (plus de duplication RAM par rang), `num_workers=4` qui accélèrent la lecture disque.
> - **Disque** : les `.pt` (~30 Go) vont sur `/kaggle/temp` (scratch, non compté dans les 20 Go de `/kaggle/working`) ; les **checkpoints** vont sur `/kaggle/working` (persistants).
> - **Sessions 12 h** : `/kaggle/temp` est éphémère → les `.pt` sont re-streamés à chaque session (streaming disque-safe), mais l'entraînement **reprend** depuis le dernier checkpoint (`ckpt_path=last`). Pour éviter le re-téléchargement, crée un *Kaggle Dataset* avec les `.pt` (voir la cellule de téléchargement).
> - Conserve toutes les optis d'entraînement : AdamW+cosine+warmup, EMA, `torch.compile`, TF32, validation allégée, modèle réduit 47,2 M.


> **Entraînement optimisé** — appliqué au-dessus de la version précédente :
>
> 1. **AdamW + scheduler cosine avec warmup** (avant : Adam à LR constant) — patch `configure_optimizers`.
> 2. **EMA des poids réellement câblée** : les poids sauvegardés dans le `.ckpt` sont les poids EMA (l'éval les charge automatiquement).
> 3. **torch.compile** + **TF32** → +20-40% de débit sur GPU récent.
> 4. **Validation allégée** : `limit_val_batches=0.25`, `val_freq=2`, 8 images d'échantillon → époques plus courtes, checkpoints plus fréquents.
> 5. **Micro-batch 1-GPU 4 → 16** → ~4× moins d'itérations par époque.
> 6. Le checkpointing robuste (`save_last`, `save_top_k=3`, reprise via `ckpt_path`) était **déjà** présent dans `trainer.py` ; l'échec d'éval précédent venait d'un système de fichiers non persistant entre sessions, pas de la config.
>
> Réglages dans la cellule YAML §7 ; patches `trainer.py` dans la cellule dédiée (après les patches fonctionnels). Sorties nettoyées.


> **Version optimisée** — corrections & réduction de paramètres appliquées :
>
> 1. **Bug corrigé** : ligne d'install `xformers` avec placeholder littéral (plantait la cellule §0).
> 2. **Paramètres réduits ~54 %** (102,4 M → 47,2 M) via `model_args` (§7) : `transformer_num_layers=2`, `num_res_blocks=[2,2,2,1,1]`. Réversible.
> 3. **Cellule de vérification** du nombre de paramètres ajoutée après la config §7.
> 4. **Doublons supprimés** : 2e `SplitManager`, 2e cellule DIAGNOSTIC, cellules vides.
> 5. **Sorties nettoyées** (notebook allégé). Relancez les cellules pour régénérer.


# synT1CE · 12 · MOTFM — FINAL EVALUATION (Samia's recommendations integrated)

**Changes vs previous version, per Samia's email:**
1. **Full test-set evaluation** — 2,573 slices (not 120) → fixes EVarΔ n=1 problem
2. **NFE = 100** (FAST-DDPM style minimum) instead of NFE=10
3. **3D TIFF export** — GT + MOTFM-predicted volumes for ≥20 patients, organised as `predictions/BraTS-MEN-XXXXX-000/{t1ce_gt.tif, t1ce_motfm_pred.tif}`
4. **Clean metrics table** (Yazdani et al. MICCAI 2025 style)
5. **OOD inference pipeline** — apply trained MOTFM to BraTS-PEDs (no fine-tuning)

> MOTFM: Yazdani et al., MICCAI 2025. Repo: https://github.com/milad1378yz/MOTFM

## 0. Installation

In [1]:
import os
if not os.path.isdir("MOTFM"):
    import subprocess; subprocess.run(["git","clone","https://github.com/milad1378yz/MOTFM.git"])
else: print("[Skip] already cloned")
%cd MOTFM
%pip install -e .
%pip install -q flow_matching torchdyn
%cd ..

import torch, sys, subprocess
V = torch.__version__            # ex. 2.5.1+cu124
base = V.split('+')[0]
cu   = V.split('+cu')[-1] if '+cu' in V else '124'
print(f"Kernel torch: {V} → Reinstalling torch & torchvision (fixing HASH MISMATCH and NMS error)…")

# Using --no-cache-dir and --force-reinstall to bypass the previous hash mismatch error
# We specifically target the cuXX index to ensure the C++ extensions (NMS) are included.
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "--no-cache-dir",
                f"torch=={base}", f"torchvision", "--index-url", f"https://download.pytorch.org/whl/cu{cu}"], check=False)

print("Reinstallation complete. Please RESTART THE SESSION (Runtime -> Restart session) and then run cell §10.")

Cloning into 'MOTFM'...


/kaggle/working/MOTFM
Obtaining file:///kaggle/working/MOTFM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 5.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of monai to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.7/48.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
motfm 0.1.0 requires numpy==1.26.4, but you have numpy 2.4.4 which is incompatible.
motfm 0.1.0 requires torch==2.5.1, but you have torch 2.10.0+cu128 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 w

Reinstallation complete. Please RESTART THE SESSION (Runtime -> Restart session) and then run cell §10.


In [2]:
# ── FIX: torchvision::nms ABI mismatch (run BEFORE the inference cell) ──────
# Cause: torch and torchvision versions are out of sync — importing `trainer`
# pulls in torchvision, whose compiled ops don't match the installed torch.
import torch, subprocess, sys

print("Current versions:")
print(f"  torch       = {torch.__version__}")
try:
    import torchvision
    print(f"  torchvision = {torchvision.__version__}")
except Exception as e:
    print(f"  torchvision import FAILED: {e}")

# Determine the matching torchvision for the installed torch.
# Compatibility map (torch → torchvision):
#   2.5.x → 0.20.x   |   2.4.x → 0.19.x   |   2.3.x → 0.18.x
#   2.2.x → 0.17.x   |   2.1.x → 0.16.x   |   2.6.x → 0.21.x
_tv_map = {
    "2.6": "0.21.0", "2.5": "0.20.0", "2.4": "0.19.0",
    "2.3": "0.18.0", "2.2": "0.17.0", "2.1": "0.16.0",
}
_tkey = ".".join(torch.__version__.split(".")[:2])
_tv_target = _tv_map.get(_tkey)

if _tv_target is None:
    print(f"\n⚠ Unknown torch {torch.__version__} — pin torchvision manually.")
else:
    # Reinstall the matching torchvision WITHOUT touching torch.
    print(f"\nReinstalling torchvision=={_tv_target} to match torch {torch.__version__}...")
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        f"torchvision=={_tv_target}",
        "--no-deps",                       # ← critical: do NOT let pip upgrade torch
        "--force-reinstall",
    ], check=False)
    print("Done. ⚠ RESTART THE RUNTIME NOW, then re-run from this cell onward.")
    print("   (Runtime → Restart session — required for the new .so to load)")

Current versions:
  torch       = 2.10.0+cu128
  torchvision import FAILED: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

⚠ Unknown torch 2.10.0+cu128 — pin torchvision manually.


In [3]:
# ── FIX: PIL._typing._Ink ImportError (run FIRST, then RESTART runtime) ──────
# Cause: broken/mismatched Pillow install — torchvision imports PIL.ImageText
# which needs a symbol (_Ink) only present in a consistent Pillow build.
import subprocess, sys

# Force a clean, consistent Pillow. 10.4.0 has _Ink and is broadly compatible
# with the torchvision versions on Kaggle/Colab.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "--force-reinstall", "--no-cache-dir",
    "Pillow==10.4.0",
], check=False)

print("Pillow reinstalled. ⚠ RESTART THE RUNTIME NOW, then run from the top.")
print("   (Runtime → Restart session — required to reload the C extension)")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 197.2 MB/s eta 0:00:00
Pillow reinstalled. ⚠ RESTART THE RUNTIME NOW, then run from the top.
   (Runtime → Restart session — required to reload the C extension)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [4]:
# ── Installation SANS toucher à NumPy (Kaggle = NumPy 2.x déjà en place) ──────
# datasets / huggingface_hub sont DÉJÀ installés sur Kaggle : on ne les réinstalle
# PAS (ça rétrograderait NumPy → rupture d'ABI avec pandas). On ajoute seulement
# nibabel, en gelant NumPy sur la version fournie par l'image Kaggle.
import subprocess, sys, numpy
NP = numpy.__version__
print("Installation en gelant numpy ==", NP)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nibabel", f"numpy=={NP}"], check=False)
import pandas as _pd, datasets as _ds
print("OK — numpy", numpy.__version__, "| pandas", _pd.__version__, "| datasets", _ds.__version__)

Installation en gelant numpy == 2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 91.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
motfm 0.1.0 requires numpy==1.26.4, but you have numpy 2.0.2 which is incompatible.
motfm 0.1.0 requires torch==2.5.1, but you have torch 2.10.0+cu128 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
cupy-cuda12x 14.0.1 requires cuda-pathfinder==1.*,>=1.3.3, but you have cuda-pathfinder 1.2.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, bu

OK — numpy 2.0.2 | pandas 2.3.3 | datasets 5.0.0


## 1. Imports & Configuration

In [5]:
from __future__ import annotations

import csv
import json
import os
import random
import sys
import warnings
import zipfile
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity as skimage_ssim
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "figure.facecolor": "white"})

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
torch.set_float32_matmul_precision("high")

print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU      : {p.name}  |  VRAM: {p.total_memory / 1e9:.1f} GB")

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : Tesla T4  |  VRAM: 15.6 GB


In [6]:
@dataclass
class Config:
    # ── Paths ────────────────────────────────────────────────────────────
    # ── Données (.pt) : gros disque scratch ÉPHÉMÈRE (recréé à chaque session) ──
    data_base        : Path  = (
        Path("/kaggle/temp") if Path("/kaggle/temp").exists() else
        Path("/content")     if Path("/content").exists() else
        Path("/root")
    )
    # ── Checkpoints / sorties : /kaggle/working (20 Go, PERSISTE entre sessions) ──
    persist_base     : Path  = (
        Path("/kaggle/working") if Path("/kaggle/working").exists() else
        Path("/content")        if Path("/content").exists() else
        Path("/root")
    )
    data_subpath     : str   = "training_data/BraTS-MEN-Train"
    tensors_name     : str   = "BraTS_2D_Tensors"
    out_name         : str   = "run_motfm_full_100pct"
    # Kaggle Dataset de .pt monté en LECTURE SEULE (recommandé, persistant, 0 disque)
    kaggle_input_dir : str   = ""    # ex: "/kaggle/input/synt1ce-brats-men-tensors"

    # ── Slice selection ──────────────────────────────────────────────────
    z_range          : Tuple[int, int] = (15, 140)
    min_brain        : float           = 0.05
    norm_mode        : Literal["zscore", "percentile"] = "percentile"

    # ── Splits ──────────────────────────────────────────────────────────
    train_frac       : float = 0.80
    val_frac         : float = 0.10
    split_seed       : int   = 42
    dataset_fraction : float = 1.0  # Configuré pour 100%

    # ── Training ────────────────────────────────────────────────────────
    max_epochs       : int   = 50
    batch_size       : int   = 16
    lr_g             : float = 2e-4
    val_every        : int   = 5
    patience         : int   = 15
    num_workers      : int   = 6
    grad_clip        : float = 1.0
    ema_decay        : float = 0.999
    modality_dropout: float = 0.5   # proba de drop de modalité(s) par échantillon train

    # ── Metrics / MOTFM ─────────────────────────────────────────────────
    metric_data_range: float = 2.0
    nfe_eval_list    : list  = None

    def __post_init__(self):
        if self.nfe_eval_list is None:
            self.nfe_eval_list = [1, 2, 5, 10, 25, 50, 100]

    @property
    def project_root(self) -> Path: return self.data_base        # compat cellules existantes
    @property
    def data_root(self)   -> Path: return self.data_base / self.data_subpath
    @property
    def tensors_dir(self) -> Path:
        # priorité au Dataset Kaggle monté (persistant, read-only), sinon scratch éphémère
        return Path(self.kaggle_input_dir) if self.kaggle_input_dir \
               else (self.data_base / self.tensors_name)
    @property
    def slice_index_csv(self) -> Path:
        # si l'index est fourni dans le Dataset monté, on le lit ; sinon on l'écrit
        # sur persist_base (car /kaggle/input est en LECTURE SEULE)
        if self.kaggle_input_dir:
            _baked = self.tensors_dir / "slice_index.csv"
            return _baked if _baked.exists() else (self.persist_base / "slice_index.csv")
        return self.tensors_dir / "slice_index.csv"
    @property
    def splits_csv(self)  -> Path:
        if self.kaggle_input_dir:
            _baked = self.tensors_dir / "splits.csv"
            if _baked.exists(): return _baked
        return self.persist_base / "splits.csv"
    @property
    def out_dir(self)     -> Path: return self.persist_base / "outputs" / self.out_name  # PERSISTE

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

print("─" * 60)
print(f"  dataset_fraction : {CFG.dataset_fraction:.0%}")
print(f"  out_dir          : {CFG.out_dir}")
print("─" * 60)

────────────────────────────────────────────────────────────
  dataset_fraction : 100%
  out_dir          : /kaggle/working/outputs/run_motfm_full_100pct
────────────────────────────────────────────────────────────


## 4. Splits (80/10/10)

In [7]:
# ── Données : Kaggle Dataset monté (idéal) OU archive HF locale (disque-safe) ──
# - Si CFG.kaggle_input_dir pointe un Dataset de .pt monté → lecture directe, 0 disque.
# - Sinon → on télécharge l'ARCHIVE via hf_hub_download (fichier LOCAL, donc PAS de
#   bug "Cannot seek" du streaming) puis on extrait ; pic disque ≈ archive + .pt (~45 Go,
#   tient dans le scratch Kaggle). Le streaming datasets et le non-streaming (~60 Go)
#   sont volontairement évités.
import os, glob, shutil, tarfile
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "krohn/synT1CE-BraTS-MEN-Tensors"
def _n_pt(d): return len(list(Path(d).glob("*.pt")))

if CFG.kaggle_input_dir and Path(CFG.kaggle_input_dir).exists():
    n = _n_pt(CFG.kaggle_input_dir)
    print(f"[Kaggle Dataset] {n:,} .pt lus directement depuis {CFG.kaggle_input_dir}")
    print("→ Aucun téléchargement, aucun disque consommé. Mode recommandé.")
    assert n > 0, "Dataset monté mais aucun .pt — vérifie le chemin / l'attachement."
else:
    CFG.tensors_dir.mkdir(parents=True, exist_ok=True)
    if _n_pt(CFG.tensors_dir) >= 60000:
        print(f"[Skip] {_n_pt(CFG.tensors_dir):,} .pt déjà présents cette session.")
    else:
        tok  = os.environ.get("HF_TOKEN") or None
        free = shutil.disk_usage(str(CFG.data_base)).free / 1e9
        print(f"Pas de Kaggle Dataset monté → téléchargement HF. Disque libre : {free:.1f} Go")
        if not tok:
            print("  (Astuce débit : ajoute un secret HF_TOKEN dans Add-ons → Secrets.)")
        # 1) trouve l'archive (100pct par défaut ; sinon 1re archive du repo)
        pct = int(CFG.dataset_fraction * 100)
        archive = f"brats_men_tensors_{pct}pct.tar.gz"
        try:
            files = list_repo_files(HF_REPO, repo_type="dataset", token=tok)
            if archive not in files:
                cands = [f for f in files if f.endswith((".tar.gz", ".tgz", ".tar"))]
                assert cands, f"Aucune archive .tar(.gz) dans {HF_REPO} : {files[:12]}"
                archive = cands[0]
            print(f"Archive : {archive}")
        except Exception as e:
            print(f"  [info] listage impossible ({e}) — on tente {archive}")
        # 2) téléchargement robuste d'un fichier LOCAL (reprise auto, pas de seek HS)
        local = hf_hub_download(HF_REPO, archive, repo_type="dataset", token=tok,
                                local_dir=str(CFG.data_base / "_hf_dl"))
        # 3) extraction défensive (aplatit toute structure interne) → tensors_dir
        print("Extraction …")
        tmpx = CFG.data_base / "_extract"; tmpx.mkdir(parents=True, exist_ok=True)
        mode = "r:gz" if str(local).endswith((".gz", ".tgz")) else "r:"
        with tarfile.open(local, mode) as tar:
            tar.extractall(str(tmpx))
        moved = 0
        for p in glob.glob(str(tmpx / "**" / "*.pt"), recursive=True):
            dst = CFG.tensors_dir / Path(p).name
            if not dst.exists():
                shutil.move(p, dst); moved += 1
        print(f"  {moved:,} .pt extraits → {CFG.tensors_dir}")
        # 4) récupère l'espace (archive + tmp d'extraction)
        shutil.rmtree(tmpx, ignore_errors=True)
        shutil.rmtree(CFG.data_base / "_hf_dl", ignore_errors=True)
        # splits.csv officiel (best-effort → persist_base ; sinon SplitManager régénère)
        try:
            sp = hf_hub_download(HF_REPO, "splits.csv", repo_type="dataset", token=tok,
                                 local_dir=str(CFG.data_base / "_hf_dl2"))
            CFG.persist_base.mkdir(parents=True, exist_ok=True)
            (CFG.persist_base / "splits.csv").write_bytes(Path(sp).read_bytes())
            shutil.rmtree(CFG.data_base / "_hf_dl2", ignore_errors=True)
        except Exception:
            pass
        print(f"Total : {_n_pt(CFG.tensors_dir):,} .pt sur {CFG.tensors_dir}")

Pas de Kaggle Dataset monté → téléchargement HF. Disque libre : 1084.2 Go
  (Astuce débit : ajoute un secret HF_TOKEN dans Add-ons → Secrets.)


Archive : brats_men_tensors_20pct.tar.gz


brats_men_tensors_20pct.tar.gz:   0%|          | 0.00/2.10G [00:00<?, ?B/s]

Extraction …
  24,401 .pt extraits → /content/BraTS_2D_Tensors


splits.csv: 0.00B [00:00, ?B/s]

Total : 24,401 .pt sur /content/BraTS_2D_Tensors


In [8]:
# ── (OPTIONNEL, à faire UNE SEULE FOIS) Créer un Kaggle Dataset des .pt ───────
# But : PERSISTER les données pour ne PLUS jamais les retélécharger. Après création,
# attache ce Dataset au notebook (Add Input), renseigne CFG.kaggle_input_dir, et la
# cellule de téléchargement lira directement /kaggle/input (0 disque, 0 download).
MAKE_KAGGLE_DATASET = False                              # ← passe à True une seule fois
KAGGLE_DS_SLUG      = "synt1ce-brats-men-tensors"        # slug de ton futur Dataset

if MAKE_KAGGLE_DATASET and not CFG.kaggle_input_dir:
    import json, subprocess
    from pathlib import Path
    # embarque aussi index + splits → Dataset autosuffisant
    for extra in (CFG.slice_index_csv, CFG.splits_csv):
        try:
            if Path(extra).exists():
                (CFG.tensors_dir / Path(extra).name).write_bytes(Path(extra).read_bytes())
        except Exception:
            pass
    user = os.environ.get("KAGGLE_USERNAME", "<ton_username>")
    meta = {"title": KAGGLE_DS_SLUG, "id": f"{user}/{KAGGLE_DS_SLUG}",
            "licenses": [{"name": "CC0-1.0"}]}
    (CFG.tensors_dir / "dataset-metadata.json").write_text(json.dumps(meta))
    print("Upload du Dataset (~30 Go, peut être long)…")
    r = subprocess.run(["kaggle", "datasets", "create", "-p", str(CFG.tensors_dir),
                        "--dir-mode", "zip"], capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    print(f"\n→ Ensuite : Add Input → ton Dataset, puis mets dans la cellule Config :")
    print(f"     CFG.kaggle_input_dir = '/kaggle/input/{KAGGLE_DS_SLUG}'")
else:
    print("[Option persistance] rien à faire (MAKE_KAGGLE_DATASET=False ou déjà monté).")

[Option persistance] rien à faire (MAKE_KAGGLE_DATASET=False ou déjà monté).


In [9]:
# ── Vérif format + cohérence splits ↔ fichiers ───────────────────────────────
import torch, glob
files = sorted(glob.glob(str(CFG.tensors_dir / "*.pt")))
assert files, "Aucun .pt trouvé — vérifie CFG.tensors_dir."
rec = torch.load(files[0], map_location="cpu")
print("type:", type(rec).__name__,
      "| clés:", list(rec.keys()) if isinstance(rec, dict) else "PAS un dict")
if isinstance(rec, dict) and {"x", "y"} <= set(rec):
    print("x:", tuple(rec["x"].shape), "| y:", tuple(rec["y"].shape))   # attendu (3,H,W)/(1,H,W)
else:
    print("⚠ Format inattendu : torch.load ne renvoie pas {x, y}. "
          "Le payload WebDataset n'était pas des octets .pt bruts — à investiguer.")

# patients réellement présents vs splits.csv
present = {Path(f).name.rsplit("_z", 1)[0] for f in files}
print(f"Patients distincts sur disque : {len(present)}")

type: dict | clés: ['x', 'y', 'seg', 'pid', 'z', 'has_tumour', 'tumour_frac']
x: (3, 240, 240) | y: (1, 240, 240)
Patients distincts sur disque : 208


In [10]:
class SplitManager:
    """
    Creates and loads patient-level 80 / 10 / 10 splits.

    Supports `dataset_fraction` to use only a percentage of each split
    for smoke-testing — the selected patients are drawn with a fixed seed
    so results are reproducible.
    """

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg

    def _all_patient_ids(self) -> List[str]:
        """Discover patient IDs from the tensor directory."""
        files    = list(self.cfg.tensors_dir.glob("*.pt"))
        if not files:
            # Fallback: discover from raw data directory
            if self.cfg.data_root.exists():
                return sorted([d.name for d in self.cfg.data_root.iterdir() if d.is_dir()])
            return []
        return sorted({f.name.rsplit("_z", 1)[0] for f in files})

    def make(self, force: bool = False) -> None:
        """
        Generate splits.csv if it does not exist yet.
        Uses CFG.split_seed for reproducibility.
        Set force=True to regenerate even if the file exists.
        """
        if self.cfg.splits_csv.exists() and not force:
            print(f"[SplitManager] splits.csv already exists: {self.cfg.splits_csv}")
            return

        pids = self._all_patient_ids()
        if not pids:
            print("[SplitManager] No patients found — run DataPreparer first.")
            return

        rng  = np.random.default_rng(self.cfg.split_seed)
        pids = list(rng.permutation(pids))
        n    = len(pids)
        t    = int(n * self.cfg.train_frac)
        v    = int(n * (self.cfg.train_frac + self.cfg.val_frac))

        rows = (
            [{"patient_id": p, "split": "train"} for p in pids[:t]]
          + [{"patient_id": p, "split": "val"}   for p in pids[t:v]]
          + [{"patient_id": p, "split": "test"}  for p in pids[v:]]
        )
        pd.DataFrame(rows).to_csv(self.cfg.splits_csv, index=False)
        print(f"[SplitManager] Written {len(pids)} patients → {self.cfg.splits_csv}")
        print(f"  train: {t} | val: {v-t} | test: {n-v}")

    def load(self) -> Dict[str, List[str]]:
        """
        Load splits.csv and apply dataset_fraction to each split.

        When dataset_fraction < 1.0, a deterministic subset of patients is
        selected from each split — small enough to run the whole pipeline in
        minutes for validation purposes.
        """
        if not self.cfg.splits_csv.exists():
            print("[SplitManager] splits.csv not found — calling make() first.")
            self.make()

        df     = pd.read_csv(self.cfg.splits_csv)
        splits = defaultdict(list)
        for _, row in df.iterrows():
            splits[row["split"]].append(row["patient_id"])

        frac = self.cfg.dataset_fraction
        if frac < 1.0:
            rng = np.random.default_rng(self.cfg.split_seed + 1)
            for key in splits:
                pids   = list(rng.permutation(splits[key]))
                k      = max(1, int(len(pids) * frac))
                splits[key] = pids[:k]
            print(
                f"[SplitManager] dataset_fraction={frac:.0%}  →  "
                f"train={len(splits['train'])} | val={len(splits['val'])} | test={len(splits['test'])} patients"
            )
        else:
            print(
                f"[SplitManager] Full dataset  →  "
                f"train={len(splits['train'])} | val={len(splits['val'])} | test={len(splits['test'])} patients"
            )
        return dict(splits)


split_mgr = SplitManager(CFG)
split_mgr.make()        # ← run once to generate splits.csv
splits = split_mgr.load()


[SplitManager] splits.csv already exists: /kaggle/working/splits.csv
[SplitManager] Full dataset  →  train=800 | val=100 | test=100 patients


In [11]:
# ── Régénération du slice_index.csv (has_tumour) — routé + skip si déjà fourni ─
import torch, pandas as pd, numpy as np
from tqdm import tqdm

_idx_path = CFG.slice_index_csv

if _idx_path.exists() and CFG.kaggle_input_dir:
    # index déjà fourni dans le Dataset Kaggle (read-only) → pas de régénération
    idx = pd.read_csv(_idx_path)
    print(f"[Skip] index déjà présent : {_idx_path} ({len(idx):,} tranches)")
else:
    pt_files   = sorted(CFG.tensors_dir.glob("*.pt"))
    threshold  = 0.8   # seuil pour isoler le cœur de la tumeur
    min_pixels = 200   # surface minimale (ignore le bruit des vaisseaux)
    rows = []
    for f in tqdm(pt_files, desc="Regen Index"):
        rec = torch.load(f, map_location="cpu", weights_only=True)
        pid = f.stem.rsplit("_z", 1)[0]
        z   = int(f.stem.rsplit("_z", 1)[1])
        x = rec["x"][0].numpy()
        y = rec["y"][0].numpy()
        num_voxels = ((y - x) > threshold).sum()
        rows.append({
            "pid": pid, "z": z, "file": f.name,
            "has_tumour":  int(num_voxels > min_pixels),
            "tumour_frac": float(num_voxels / (240 * 240)),
        })
    idx = pd.DataFrame(rows)
    _idx_path.parent.mkdir(parents=True, exist_ok=True)
    idx.to_csv(_idx_path, index=False)
    print(f"Index finalisé : {len(idx):,} tranches, "
          f"{idx['has_tumour'].sum():,} avec tumeur ({idx['has_tumour'].mean():.1%}).")

Regen Index: 100%|██████████| 24401/24401 [00:41<00:00, 584.49it/s]


Index finalisé : 24,401 tranches, 9,829 avec tumeur (40.3%).


## 5. Load Index

In [12]:
_idx_path = CFG.slice_index_csv
if not _idx_path.exists():
    raise FileNotFoundError(
        f"slice_index.csv introuvable → relancez la cellule DataPreparer (§3) d'abord.\n"
        f"  Attendu : {_idx_path}"
    )
if not CFG.splits_csv.exists():
    raise FileNotFoundError(
        f"splits.csv introuvable → relancez la cellule SplitManager (§4) d'abord.\n"
        f"  Attendu : {CFG.splits_csv}"
    )

idx       = pd.read_csv(_idx_path)
splits_df = pd.read_csv(CFG.splits_csv)

def get_split(split_name, fraction=None):
    pids = splits_df.loc[splits_df["split"] == split_name, "patient_id"].tolist()
    if fraction and fraction < 1.0:
        rng  = np.random.default_rng(42)
        pids = list(rng.permutation(sorted(pids)))[:max(1, int(len(pids) * fraction))]
    return idx[idx["pid"].isin(set(pids))].reset_index(drop=True)

f         = CFG.dataset_fraction
train_idx = get_split("train", f)
val_idx   = get_split("val",   f)
test_idx  = get_split("test",  f)
print(f"train={len(train_idx):,} val={len(val_idx):,} test={len(test_idx):,} slices")

train=19,109 val=2,719 test=2,573 slices


## 6. Metrics Suite (Samia's requirements — EVarΔ / Wasserstein / Multimodality)

In [13]:
# ═══════════════════════════════════════════════════════════════════════════════
# synT1CE — Shared Metrics Suite (Samia's requirements, July 2026)
# Paste this block into any notebook's evaluation section.
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import wasserstein_distance
from skimage.metrics import structural_similarity as skimage_ssim
from skimage.metrics import peak_signal_noise_ratio  as skimage_psnr
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── 1.  Enhancement region (intensity-derived, no GT leakage) ────────────────
def enhancement_mask(t1ce: np.ndarray,
                     t1n:  np.ndarray,
                     threshold: float = 0.1) -> np.ndarray:
    """
    Intensity-derived enhancement region: voxels where (T1CE - T1n) > threshold
    after [-1,1] normalisation.  Does NOT use the BraTS ET/tumour label,
    so it is safe to use at inference without GT leakage.
    """
    return (t1ce - t1n) > threshold


# ── 2.  Save per-slice predictions ──────────────────────────────────────────
def save_predictions(model, test_idx: pd.DataFrame, tensors_dir: Path,
                     out_dir: Path, device, data_range: float = 2.0,
                     ema_model=None) -> None:
    """
    Run inference on the test split, save each prediction as a .npy file,
    and return a MetricsTracker populated with SSIM/PSNR/MAE.

    out_dir/<pid>/<slice_stem>.npy  — shape (H, W), float32
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    G = ema_model if ema_model is not None else model
    G.eval()

    for _, row in tqdm(test_idx.iterrows(), total=len(test_idx),
                        desc="Saving predictions", ncols=80):
        rec  = torch.load(tensors_dir / row["file"],
                           map_location="cpu", weights_only=True)
        x    = rec["x"].unsqueeze(0).to(device)
        with torch.no_grad():
            pred = G(x).squeeze().cpu().numpy()   # (H, W)

        pid_dir = out_dir / str(row["pid"])
        pid_dir.mkdir(parents=True, exist_ok=True)
        np.save(pid_dir / (Path(row["file"]).stem + ".npy"), pred.astype(np.float32))

    print(f"Predictions saved -> {out_dir}  ({len(test_idx)} slices)")


# ── 3.  EVarΔ — Enhancement Variance metric ──────────────────────────────────
def compute_evar_delta(test_idx: pd.DataFrame,
                        pred_dir: Path,
                        tensors_dir: Path,
                        threshold: float = 0.1) -> dict:
    """
    EVarΔ(p) = Var_{z}(mean_enhancement_pred_z) / Var_{z}(mean_enhancement_gt_z)

    Returns per-patient dict and a summary dict.
    EVarΔ = 1 → perfect volumetric coherence
    EVarΔ < 1 → variance contraction (L1 regression-to-mean failure mode)
    EVarΔ > 1 → variance inflation
    """
    per_patient = {}

    for pid, group in test_idx[test_idx["has_tumour"] == True].groupby("pid"):
        group = group.sort_values("z")
        delta_pred_means, delta_gt_means = [], []

        for _, row in group.iterrows():
            npy_path = pred_dir / str(pid) / (Path(row["file"]).stem + ".npy")
            if not npy_path.exists():
                continue
            pred = np.load(npy_path)
            rec  = torch.load(tensors_dir / row["file"],
                               map_location="cpu", weights_only=True)
            t1n    = rec["x"][0].numpy()
            t1ce   = rec["y"][0].numpy()

            enh_mask = enhancement_mask(t1ce, t1n, threshold)
            if enh_mask.sum() < 10:
                continue

            delta_pred_means.append(float((pred  - t1n)[enh_mask].mean()))
            delta_gt_means.append(  float((t1ce  - t1n)[enh_mask].mean()))

        if len(delta_gt_means) < 2:
            continue

        var_pred = float(np.var(delta_pred_means, ddof=1))
        var_gt   = float(np.var(delta_gt_means,   ddof=1))
        if var_gt < 1e-8:
            continue

        per_patient[pid] = var_pred / var_gt

    vals = list(per_patient.values())
    if not vals:
        return {"per_patient": {}, "mean": None, "std": None,
                "median": None, "pct_lt1": None, "n": 0}

    return {
        "per_patient": per_patient,
        "mean":    float(np.mean(vals)),
        "std":     float(np.std(vals)),
        "median":  float(np.median(vals)),
        "pct_lt1": float(np.mean(np.array(vals) < 1.0) * 100),
        "n":       len(vals),
    }


# ── 4.  Wasserstein-1 distance on enhancement distributions ─────────────────
def compute_wasserstein(test_idx: pd.DataFrame,
                         pred_dir: Path,
                         tensors_dir: Path,
                         threshold: float = 0.1) -> dict:
    """
    Wasserstein-1 distance between the pooled predicted and GT enhancement
    distributions over all tumour-region voxels in the test set.

    d_W = integral |F_GT(t) - F_pred(t)| dt
    """
    all_pred_vals, all_gt_vals = [], []

    for _, row in tqdm(test_idx[test_idx["has_tumour"] == True].iterrows(),
                        desc="Wasserstein", ncols=80):
        npy_path = pred_dir / str(row["pid"]) / (Path(row["file"]).stem + ".npy")
        if not npy_path.exists():
            continue
        pred = np.load(npy_path)
        rec  = torch.load(tensors_dir / row["file"],
                           map_location="cpu", weights_only=True)
        t1n  = rec["x"][0].numpy()
        t1ce = rec["y"][0].numpy()

        enh_mask = enhancement_mask(t1ce, t1n, threshold)
        if enh_mask.sum() < 5:
            continue

        # Enhancement maps: pred - T1n and GT - T1n
        all_pred_vals.extend((pred  - t1n)[enh_mask].tolist())
        all_gt_vals.extend(  (t1ce  - t1n)[enh_mask].tolist())

    if not all_gt_vals:
        return {"d_W": None, "n_voxels": 0}

    # Sample for speed (Wasserstein is O(N log N))
    rng = np.random.default_rng(42)
    N   = min(len(all_gt_vals), 100_000)
    gt_s   = rng.choice(all_gt_vals,   size=N, replace=False)
    pred_s = rng.choice(all_pred_vals, size=N, replace=False)

    d_w = float(wasserstein_distance(gt_s, pred_s))
    return {"d_W": d_w, "n_voxels": len(all_gt_vals)}


# ── 5.  Multimodality proof (inter-patient variance with matched inputs) ─────
def prove_multimodality(train_idx: pd.DataFrame,
                         tensors_dir: Path,
                         n_pairs: int = 200,
                         epsilon: float = 0.01,
                         threshold: float = 0.1) -> dict:
    """
    Empirical proof that p(T1CE | x) is multimodal over tumour voxels.

    For n_pairs randomly sampled patient pairs (i, j) from the TRAINING set:
    1. Compute mean-squared difference between their non-contrast inputs x_i, x_j
    2. Keep pair if MSE(x_i, x_j) < epsilon  (inputs "matched")
    3. Compute ||Delta*_i - Delta*_j||_1 in the enhancement region
    If p(T1CE|x) were unimodal, matched inputs would give similar T1CE.
    High ||Delta*_i - Delta*_j|| for matched pairs = empirical multimodality.
    """
    tumour_pids = list(
        train_idx[train_idx["has_tumour"] == True]["pid"].unique()
    )
    rng = np.random.default_rng(42)

    matched_diffs, all_input_msds = [], []
    attempts, matched = 0, 0

    while matched < n_pairs and attempts < n_pairs * 20:
        attempts += 1
        pid_i, pid_j = rng.choice(tumour_pids, size=2, replace=False)
        # Pick one tumour slice from each
        rows_i = train_idx[(train_idx["pid"] == pid_i) &
                            (train_idx["has_tumour"] == True)]
        rows_j = train_idx[(train_idx["pid"] == pid_j) &
                            (train_idx["has_tumour"] == True)]
        if rows_i.empty or rows_j.empty:
            continue

        row_i = rows_i.sample(1, random_state=int(attempts)).iloc[0]
        row_j = rows_j.sample(1, random_state=int(attempts)+1).iloc[0]

        try:
            rec_i = torch.load(tensors_dir / row_i["file"],
                                map_location="cpu", weights_only=True)
            rec_j = torch.load(tensors_dir / row_j["file"],
                                map_location="cpu", weights_only=True)
        except Exception:
            continue

        x_i, y_i = rec_i["x"].numpy(), rec_i["y"][0].numpy()
        x_j, y_j = rec_j["x"].numpy(), rec_j["y"][0].numpy()

        msd = float(np.mean((x_i - x_j) ** 2))
        all_input_msds.append(msd)

        if msd > epsilon:
            continue  # inputs too different — skip

        matched += 1
        t1n_i = x_i[0]; t1n_j = x_j[0]
        enh_i = enhancement_mask(y_i, t1n_i, threshold)
        enh_j = enhancement_mask(y_j, t1n_j, threshold)
        common = enh_i | enh_j  # union of enhancement regions
        if common.sum() < 5:
            continue

        d_ij = float(np.mean(np.abs(
            (y_i - t1n_i)[common] - (y_j - t1n_j)[common]
        )))
        matched_diffs.append(d_ij)

    return {
        "n_matched_pairs": len(matched_diffs),
        "n_attempts": attempts,
        "epsilon": epsilon,
        "mean_input_msd": float(np.mean(all_input_msds)) if all_input_msds else None,
        "matched_L1_mean": float(np.mean(matched_diffs)) if matched_diffs else None,
        "matched_L1_std":  float(np.std(matched_diffs))  if matched_diffs else None,
        "matched_L1_vals": matched_diffs,
    }


# ── 6.  Full metrics summary printer ────────────────────────────────────────
def print_metrics_summary(tier1: dict, evar: dict, wass: dict,
                            model_name: str = "") -> None:
    print(f"\n{'='*60}")
    print(f"  {model_name} — Metrics Summary")
    print(f"{'='*60}")
    print("\n  Tier 1 (pixel fidelity, test set):")
    for grp in ("all", "tumour", "healthy"):
        if grp in tier1:
            g = tier1[grp]
            print(f"    {grp:8s}  SSIM={g.get('ssim',0):.4f}  "
                  f"PSNR={g.get('psnr',0):.2f}  MAE={g.get('mae',0):.4f}")
    print(f"\n  EVarΔ (volumetric enhancement coherence):")
    if evar.get("mean") is not None:
        print(f"    mean={evar['mean']:.4f}  std={evar['std']:.4f}  "
              f"median={evar['median']:.4f}  "
              f"{evar['pct_lt1']:.0f}% patients < 1  (n={evar['n']})")
    else:
        print("    EVarΔ: not yet computed (save predictions first)")
    print(f"\n  Wasserstein distance (enhancement distribution):")
    if wass.get("d_W") is not None:
        print(f"    d_W = {wass['d_W']:.4f}  "
              f"(over {wass['n_voxels']:,} tumour voxels)")
    else:
        print("    d_W: not yet computed")
    print(f"{'='*60}\n")

print("Metrics suite loaded — all functions ready.")


Metrics suite loaded — all functions ready.


In [14]:
class MetricsTracker:
    """
    Accumulates per-slice SSIM/PSNR/MAE, stratified by all/tumour/healthy.
    Brain-mask only. Fixed global data_range (matches main pipeline P2 fix).
    """

    def __init__(self, data_range: float = 2.0) -> None:
        self.data_range = data_range
        self.records: List[dict] = []

    def update(self, pred_np: np.ndarray, gt_np: np.ndarray, has_tumour: bool) -> None:
        mask = gt_np != 0
        if mask.sum() == 0:
            return
        dr = self.data_range
        self.records.append({
            "ssim":       float(skimage_ssim(pred_np, gt_np, data_range=dr)),
            "psnr":       float(skimage_psnr(gt_np, pred_np, data_range=dr)),
            "mae":        float(np.abs(pred_np[mask] - gt_np[mask]).mean()),
            "has_tumour": has_tumour,
        })

    def _agg(self, subset: List[dict], key: str) -> dict:
        vals = [r[key] for r in subset if np.isfinite(r[key])]
        if not vals:
            return {"mean": float("nan"), "std": float("nan"), "n": 0}
        return {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "n": len(vals)}

    def summary(self) -> dict:
        tumour  = [r for r in self.records if     r["has_tumour"]]
        healthy = [r for r in self.records if not r["has_tumour"]]
        return {
            "all":     {k: self._agg(self.records, k) for k in ("ssim", "psnr", "mae")},
            "tumour":  {k: self._agg(tumour,        k) for k in ("ssim", "psnr", "mae")},
            "healthy": {k: self._agg(healthy,       k) for k in ("ssim", "psnr", "mae")},
        }

print("MetricsTracker ready.")

MetricsTracker ready.


## 7. Clone & Install MOTFM

In [15]:
import yaml, torch
ref_cfg_path = Path('MOTFM/configs/mask_class_conditioning.yaml')
CKPT_DIR = str(CFG.out_dir / 'motfm_checkpoints')     # sur /kaggle/working → PERSISTE

n_gpus = torch.cuda.device_count()
print(f"GPU détectés : {n_gpus}")

# ── DDP maintenant SÛR : le Dataset paresseux ne copie PAS le dataset en RAM ──
#    (1 lecture .pt par échantillon, pas de tenseur géant dupliqué par rang)
if n_gpus > 1:
    _strategy, _devices = "ddp", n_gpus
else:
    _strategy, _devices = "auto", max(1, n_gpus)

PER_GPU_BATCH    = 16     # T4 16 Go + modèle 47M fp16 ; baisser à 8 si OOM VRAM
TARGET_EFF_BATCH = 32
_global = PER_GPU_BATCH * _devices
_accum  = max(1, round(TARGET_EFF_BATCH / max(1, _global)))
_eff    = _global * _accum

with open(ref_cfg_path) as f: ref_cfg = yaml.safe_load(f)

ref_cfg.setdefault('data_args', {}).update({
    # ── Chargement PARESSEUX (aucun pickle, aucune copie RAM du dataset) ──
    'lazy_loading':    True,
    'tensors_dir':     str(CFG.tensors_dir),
    'slice_index_csv': str(CFG.slice_index_csv),
    'splits_csv':      str(CFG.splits_csv),
    'norm_scope':      'sample',        # min-max par échantillon (compatible lazy)
        'modality_dropout': CFG.modality_dropout,   # dropout de modalités à l'entraînement
    'pickle_path':     str(CFG.out_dir / 'motfm_data.pkl'),  # inutilisé en lazy
    'split_train':'train','split_val':'val','split_test':'test','class_values':[0,1],
})
ref_cfg.setdefault('model_args', {}).update({
    'cross_attention_dim': 2, 'use_flash_attention': False,
    'transformer_num_layers': 2, 'num_res_blocks': [2, 2, 2, 1, 1],   # 47,2 M (-54%)
})
ref_cfg.setdefault('train_args', {}).update({
    'checkpoint_dir': CKPT_DIR,
    'num_epochs': CFG.max_epochs,
    'batch_size': PER_GPU_BATCH,
    'lr': CFG.lr_g,
    'num_workers': 4,          # lazy → les workers ACCÉLÈRENT la lecture disque
    'pin_memory': True,
    'persistent_workers': True,
    'prefetch_factor': 4,
    'gradient_accumulation_steps': _accum,
    'grad_clip_norm': 1.0,
    'accelerator': 'gpu', 'devices': _devices, 'strategy': _strategy,
    'precision': '16-mixed',
    # ── Validation allégée + checkpoints fréquents (persistants) ──
    'val_freq': 2, 'limit_val_batches': 0.25, 'num_val_samples': 8,
    # ── Optimiseur + scheduler + EMA + compile (patchés dans trainer.py) ──
    'weight_decay': 0.0, 'use_lr_scheduler': True, 'warmup_steps': None, 'min_lr_ratio': 0.05,
    'use_compile': True, 'use_ema': True, 'ema_decay': CFG.ema_decay,
    'log_every_n_steps': 50, 'num_sanity_val_steps': 0,
})
config_path = CFG.out_dir / 'motfm_config.yaml'
CFG.out_dir.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f: yaml.safe_dump(ref_cfg, f, sort_keys=False)
ta = ref_cfg['train_args']
print(f"Config → {config_path}")
print(f"  strategy={ta['strategy']}  devices={ta['devices']}  batch/GPU={ta['batch_size']}  "
      f"accum={ta['gradient_accumulation_steps']} → batch effectif {_eff}")
print(f"  lazy_loading=True  num_workers={ta['num_workers']}  pin_memory={ta['pin_memory']}")
print(f"  precision={ta['precision']}  epochs={ta['num_epochs']}  val_freq={ta['val_freq']}  "
      f"limit_val_batches={ta['limit_val_batches']}")
print(f"  checkpoints (PERSISTANTS) → {CKPT_DIR}")

GPU détectés : 2
Config → /kaggle/working/outputs/run_motfm_full_100pct/motfm_config.yaml
  strategy=ddp  devices=2  batch/GPU=16  accum=1 → batch effectif 32
  lazy_loading=True  num_workers=4  pin_memory=True
  precision=16-mixed  epochs=50  val_freq=2  limit_val_batches=0.25
  checkpoints (PERSISTANTS) → /kaggle/working/outputs/run_motfm_full_100pct/motfm_checkpoints


In [16]:
# ── [AJOUT] Vérification du nombre de paramètres du modèle ────────────────────
# Instancie UNet + ControlNet depuis le YAML effectif et affiche le total.
# Permet de mesurer l'effet des réglages model_args (réduction de paramètres).
import yaml
from generative.networks.nets import DiffusionModelUNet, ControlNet

with open(config_path) as _f:
    _ma = yaml.safe_load(_f)['model_args']

_cnet_keys = {'spatial_dims','in_channels','num_res_blocks','num_channels',
    'attention_levels','norm_num_groups','resblock_updown','num_head_channels',
    'with_conditioning','transformer_num_layers','cross_attention_dim',
    'use_flash_attention','norm_eps'}

def _count(m): return sum(p.numel() for p in m.parameters())

_mc = {k: v for k, v in _ma.items()
       if k not in ('conditioning_embedding_num_channels',
                    'conditioning_embedding_in_channels', 'mask_conditioning')}
_unet = DiffusionModelUNet(**_mc)
_cn_kwargs = {k: v for k, v in _mc.items() if k in _cnet_keys}
_cnet = ControlNet(
    **_cn_kwargs,
    conditioning_embedding_num_channels=_ma.get('conditioning_embedding_num_channels', [16]),
    conditioning_embedding_in_channels=_ma.get('conditioning_embedding_in_channels', 3),
)
_u, _c = _count(_unet), _count(_cnet)
print(f"  UNet        : {_u/1e6:7.2f} M")
print(f"  ControlNet  : {_c/1e6:7.2f} M")
print(f"  ─────────────────────────")
print(f"  TOTAL       : {(_u+_c)/1e6:7.2f} M paramètres entraînables")
print(f"  (baseline full = 102.35 M  →  réduction {(1-(_u+_c)/1e6/102.35)*100:+.0f}%)")
del _unet, _cnet  # libère la RAM

  UNet        :   33.48 M
  ControlNet  :   13.74 M
  ─────────────────────────
  TOTAL       :   47.22 M paramètres entraînables
  (baseline full = 102.35 M  →  réduction +54%)


In [17]:
# ── (PLUS de pickle géant) Vérification de l'index pour le chargement paresseux ─
# Le Dataset paresseux lit directement les .pt via slice_index.csv + splits.csv.
# Aucun motfm_data.pkl n'est créé : on économise ~30 Go de disque ET toute la RAM.
import pandas as pd, random
assert CFG.slice_index_csv.exists(), f"slice_index.csv manquant : {CFG.slice_index_csv}"
assert CFG.splits_csv.exists(),      f"splits.csv manquant : {CFG.splits_csv}"
_idx = pd.read_csv(CFG.slice_index_csv); _spl = pd.read_csv(CFG.splits_csv)
_pid = "patient_id" if "patient_id" in _spl.columns else _spl.columns[0]
print(f"slices indexées : {len(_idx):,} | patients : {_spl[_pid].nunique()}")
for _s in ("train", "val", "test"):
    _pids = set(_spl.loc[_spl["split"] == _s, _pid])
    print(f"  {_s:5s}: {int(_idx['pid'].isin(_pids).sum()):,} slices")
# cohérence : un .pt existe bien pour un échantillon tiré au hasard
_f = _idx["file"].iloc[random.randint(0, len(_idx) - 1)]
assert (CFG.tensors_dir / _f).exists(), f".pt manquant sur disque : {_f}"
print("✓ index cohérent avec les .pt — prêt pour le chargement paresseux (aucun pickle)")

slices indexées : 24,401 | patients : 1000
  train: 19,109 slices
  val  : 2,719 slices
  test : 2,573 slices
✓ index cohérent avec les .pt — prêt pour le chargement paresseux (aucun pickle)


In [18]:
# ── Patch MOTFM/trainer.py : strategy configurable depuis le YAML ────────────
# Par défaut trainer.py force DDPStrategy pour devices>1 (spawn → 2× RAM).
# Ce patch lit "strategy" depuis train_args → permet "dp" (DataParallel, 1 processus).

from pathlib import Path

_tr  = Path('MOTFM/trainer.py')
_src = _tr.read_text()

_old = (
    '    accelerator = tr.get("accelerator", "auto")\n'
    '    devices = tr.get("devices", "auto")\n'
    '    strategy = _resolve_strategy(accelerator=accelerator, devices=devices)'
)
_new = (
    '    accelerator = tr.get("accelerator", "auto")\n'
    '    devices = tr.get("devices", "auto")\n'
    '    _strategy_cfg = tr.get("strategy", None)\n'
    '    if _strategy_cfg is not None:\n'
    '        strategy = _strategy_cfg  # override depuis le YAML\n'
    '    else:\n'
    '        strategy = _resolve_strategy(accelerator=accelerator, devices=devices)'
)

if _old in _src:
    _src = _src.replace(_old, _new)
    print('Patch trainer.py (strategy override) ✓')
elif '_strategy_cfg' in _src:
    print('Patch déjà appliqué')
else:
    print('Pattern introuvable — vérifiez trainer.py')

_tr.write_text(_src)
print('trainer.py patché ✓')
print('Utilisation : ajouter strategy: "dp" dans train_args du YAML')


Patch trainer.py (strategy override) ✓
trainer.py patché ✓
Utilisation : ajouter strategy: "dp" dans train_args du YAML


In [19]:
# ── Patch trainer.py : autocast autour de validate_and_save_samples ──────────
# Root cause : on_validation_epoch_end s'exécute hors autocast Lightning →
#   params float32, mask float16 → mismatch dans ControlNet Linear.
# Fix : wrapper validate_and_save_samples dans autocast fp16.

from pathlib import Path

# Cherche trainer.py dans les emplacements courants (Kaggle, Modal, local)
_candidates = [
    Path('MOTFM/trainer.py'),           # Kaggle /kaggle/working/MOTFM/
    Path('/root/MOTFM/trainer.py'),     # Modal / local root
    Path('/kaggle/working/MOTFM/trainer.py'),
]
_tr = next((p for p in _candidates if p.exists()), None)
if _tr is None:
    import glob
    found = glob.glob('**/trainer.py', recursive=True)
    _tr = Path(found[0]) if found else None

if _tr is None:
    print('trainer.py introuvable — vérifiez le chemin MOTFM')
else:
    print(f'trainer.py trouvé : {_tr}')
    _src = _tr.read_text()

    if "autocast('cuda'" in _src or 'autocast("cuda"' in _src:
        print('Patch autocast déjà présent ✓')
    else:
        import re
        lines = _src.splitlines(keepends=True)
        patched = False
        for i, line in enumerate(lines):
            stripped = line.lstrip()
            if ('validate_and_save_samples(' in stripped
                    and not stripped.startswith('def ')
                    and not stripped.startswith('#')):
                ind = line[:len(line) - len(stripped)]
                lines.insert(i,
                    f"{ind}with __import__('torch').amp.autocast('cuda', "
                    f"dtype=__import__('torch').float16):\n")
                depth = 0
                j = i + 1
                while j < len(lines):
                    lines[j] = '    ' + lines[j]
                    depth += lines[j].count('(') - lines[j].count(')')
                    if depth <= 0:
                        j += 1; break
                    j += 1
                patched = True
                print(f'Patch autocast validation ✓  (inséré avant ligne {i+1})')
                break
        if not patched:
            print('Pattern validate_and_save_samples introuvable :')
            for i, l in enumerate(_src.splitlines()):
                if 'validate' in l.lower() and 'samples' in l.lower():
                    print(f'  [{i+1}] {l!r}')
        else:
            _tr.write_text(''.join(lines))
            print(f'{_tr} sauvegardé ✓')


trainer.py trouvé : MOTFM/trainer.py
Patch autocast validation ✓  (inséré avant ligne 264)
MOTFM/trainer.py sauvegardé ✓


In [20]:
# ── [FIX v2] Patch general_utils.py : données en float32 (PAS float16) ────────
# Root cause "mat1 and mat2 must have the same dtype (Half vs Float)" à l'inférence :
# caster image/mask en float16 au chargement crée un mismatch avec les poids
# float32 dans les couches Linear. Sous precision 16-mixed, l'autocast gère déjà
# la demi-précision ; on charge donc les données en float32.
import glob
from pathlib import Path

_candidates = [
    Path('MOTFM/utils/general_utils.py'),
    Path('/root/MOTFM/utils/general_utils.py'),
    Path('/kaggle/working/MOTFM/utils/general_utils.py'),
]
_gu = next((p for p in _candidates if p.exists()), None)
if _gu is None:
    found = glob.glob('**/general_utils.py', recursive=True)
    _gu = Path(found[0]) if found else None

if _gu is None:
    print('general_utils.py introuvable')
else:
    print(f'general_utils.py : {_gu}')
    _src = _gu.read_text()

    # force float32 (remplace toute variante float16 laissée par d'anciens runs)
    _reverts = [
        ('torch.as_tensor(e["image"], dtype=torch.float16)',
         'torch.as_tensor(e["image"], dtype=torch.float32)',
         'image -> float32'),
        ('torch.as_tensor(e["mask"], dtype=torch.float16)',
         'torch.as_tensor(e["mask"], dtype=torch.float32)',
         'mask -> float32'),
    ]
    for old, new, label in _reverts:
        if old in _src:
            _src = _src.replace(old, new); print(f'  {label} ✓ (reverti depuis float16)')
        elif new in _src:
            print(f'  {label} déjà OK')
        else:
            print(f'  {label} : pattern introuvable')
    _gu.write_text(_src)
    print('general_utils.py : données en float32 ✓')


# ── [FIX] Patch save_image : gère le conditioning 3 canaux (3,H,W) ───────────
# validate_and_save_samples appelle save_image(cnd_img) avec cnd_img (3,240,240)
# → plt.imshow refuse. save_image ne squeezait que (1,H,W). On collapse tout >2D.
if _gu is not None:
    _s2 = _gu.read_text()
    _marker = "# _FIX_save_image_multichannel_"
    if _marker in _s2:
        print("Patch save_image (multi-canal) déjà appliqué")
    else:
        _old_block = (
            '    if img_tensor.dim() == 3 and img_tensor.shape[0] == 1:\n'
            '        img_tensor = img_tensor.squeeze(0)\n'
            '    plt.figure()\n'
            '    plt.imshow(img_tensor.cpu().numpy(), cmap="gray")'
        )
        _new_block = (
            '    ' + _marker + '\n'
            '    import numpy as _np\n'
            '    _arr = img_tensor.detach().cpu().numpy() if hasattr(img_tensor, "detach") else _np.asarray(img_tensor)\n'
            '    _arr = _np.asarray(_arr)\n'
            '    while _arr.ndim > 2:      # collapse batch/canal (ex: conditioning 3 canaux)\n'
            '        _arr = _arr[0]\n'
            '    plt.figure()\n'
            '    plt.imshow(_arr, cmap="gray")'
        )
        if _old_block in _s2:
            _s2 = _s2.replace(_old_block, _new_block)
            _gu.write_text(_s2)
            print("Patch save_image (multi-canal) ✓")
        else:
            print("[warn] bloc save_image original introuvable")


general_utils.py : MOTFM/utils/general_utils.py
  image -> float32 déjà OK
  mask -> float32 déjà OK
general_utils.py : données en float32 ✓
Patch save_image (multi-canal) ✓


In [21]:
# Patch MOTFM/utils/utils_fm.py — reset git + 3 patches propres
import subprocess, re, py_compile, tempfile, os, glob
from pathlib import Path

# ── Résolution du chemin ─────────────────────────────────────────────────────
_candidates = [
    Path('MOTFM'),
    Path('/root/MOTFM'),
    Path('/kaggle/working/MOTFM'),
]
_motfm = next((p for p in _candidates if (p / 'utils/utils_fm.py').exists()), None)
if _motfm is None:
    found = glob.glob('**/utils_fm.py', recursive=True)
    _motfm = Path(found[0]).parent.parent if found else None

if _motfm is None:
    print('MOTFM introuvable — vérifiez le chemin')
    raise SystemExit

_fm = _motfm / 'utils/utils_fm.py'
print(f'utils_fm.py : {_fm}')

# ── 1. Reset git ──────────────────────────────────────────────────────────────
r = subprocess.run(['git','checkout','utils/utils_fm.py'], cwd=str(_motfm),
                   capture_output=True, text=True)
print('git reset :', 'OK' if r.returncode==0 else r.stderr.strip())
_src = _fm.read_text()

# ── Patch A : pop avant DiffusionModelUNet ────────────────────────────────────
m = re.search(r'^(\s+)(unet\s*=\s*DiffusionModelUNet\(\*\*mc\))', _src, re.M)
if m:
    _src = _src.replace(m.group(0),
        f"{m.group(1)}_cond_in_ch = mc.pop('conditioning_embedding_in_channels', 3)\n{m.group(0)}", 1)
    print('Patch A (pop avant UNet) ✓')
else:
    print('Patch A : pattern introuvable')

# ── Patch B : passe _cond_in_ch à ControlNet ─────────────────────────────────
m = re.search(r'^(\s+)(controlnet\s*=\s*ControlNet\(\*\*mc)', _src, re.M)
if m:
    _src = _src.replace(m.group(0),
        f"{m.group(1)}controlnet = ControlNet(**mc, conditioning_embedding_in_channels=_cond_in_ch", 1)
    print('Patch B (ControlNet in_channels=3) ✓')
else:
    print('Patch B : pattern introuvable')

# ── Patch C : cast mask → dtype modèle (maintenu pour cohérence forward) ─────
# Note : le fix principal est dans patch_val_autocast (trainer.py autocast).
# Ce patch reste utile si le forward est appelé hors autocast dans d'autres contextes.
if 'mask.to(dtype=next(self.controlnet.parameters()).dtype)' in _src:
    print('Patch C déjà appliqué')
else:
    _lines = _src.splitlines(keepends=True)
    inserted = False
    for i, line in enumerate(_lines):
        stripped = line.lstrip()
        if re.match(r'x\s*=\s*mask\s*,', stripped):
            ctx_before = ''.join(_lines[max(0, i-8):i])
            if 'self.controlnet(' in ctx_before:
                indent = line[:len(line) - len(stripped)]
                _lines[i] = f'{indent}x=mask.to(dtype=next(self.controlnet.parameters()).dtype),\n'
                inserted = True
                print(f'Patch C (mask dtype cast) ✓  — ligne {i+1}')
                break
    if not inserted:
        print('Patch C : pattern x=mask, introuvable (non bloquant — autocast fix dans trainer.py)')
    _src = ''.join(_lines)

# ── [FIX v2] Patch D : plot_solver_steps — imshow du masque 3 canaux ─────────
# 2e source du crash training : plot_solver_steps fait
#   imshow(mask_batch[i].cpu().numpy().squeeze()) → (3,240,240), squeeze() ne
#   réduit pas (aucune dim == 1) → "Invalid shape (3,240,240)". On collapse >2D.
_old_ps = 'axes[i][col].imshow(mask_batch[i].cpu().numpy().squeeze(), cmap="gray")'
_new_ps = ('axes[i][col].imshow('
           '(lambda _a: _a[0] if _a.ndim > 2 else _a.squeeze())'
           '(mask_batch[i].detach().cpu().numpy()), cmap="gray")')
if _new_ps in _src:
    print('Patch D (plot_solver_steps masque) déjà appliqué')
elif _old_ps in _src:
    _src = _src.replace(_old_ps, _new_ps)
    print('Patch D (plot_solver_steps masque 3 canaux) ✓')
else:
    print('Patch D : pattern imshow(mask_batch...) introuvable')

# ── Vérification syntaxe et écriture ─────────────────────────────────────────
tmp = tempfile.mktemp(suffix='.py')
try:
    open(tmp, 'w').write(_src)
    py_compile.compile(tmp, doraise=True)
    _fm.write_text(_src)
    print('utils_fm.py — syntaxe OK, écrit ✓')
except py_compile.PyCompileError as e:
    lines = _src.splitlines()
    lineno = 0
    try: lineno = int(str(e).split('line ')[1].split(')')[0])
    except: pass
    print(f'ERREUR syntaxe ligne {lineno}:')
    for j in range(max(0, lineno-3), min(len(lines), lineno+3)):
        print(f'  [{j+1}] {lines[j]}')
finally:
    os.unlink(tmp)


utils_fm.py : MOTFM/utils/utils_fm.py
git reset : OK
Patch A (pop avant UNet) ✓
Patch B (ControlNet in_channels=3) ✓
Patch C : pattern x=mask, introuvable (non bloquant — autocast fix dans trainer.py)
Patch D (plot_solver_steps masque 3 canaux) ✓
utils_fm.py — syntaxe OK, écrit ✓


In [22]:
# ── [OPT] Patches d'optimisation de l'entraînement (trainer.py) ──────────────
# Appliqués APRÈS les patches fonctionnels précédents. Idempotents (marqueur).
# Contenu :
#   1. TF32 (matmul haute vitesse sur Ampere/L4/A100)
#   2. configure_optimizers : Adam(LR constant) → AdamW + cosine + warmup
#   3. torch.compile du modèle (si use_compile)
#   4. limit_val_batches câblé dans le Trainer (validation allégée)
#   5. EMACallback : moyenne exponentielle des poids, ACTIVE pendant la
#      validation + la sauvegarde → le .ckpt contient directement les poids EMA
#      (l'évaluation en aval charge donc l'EMA sans modification).
import glob, py_compile, tempfile, os
from pathlib import Path

_cands = [Path('MOTFM/trainer.py'), Path('/root/MOTFM/trainer.py'),
          Path('/kaggle/working/MOTFM/trainer.py')]
_tr = next((p for p in _cands if p.exists()), None)
if _tr is None:
    _f = glob.glob('**/trainer.py', recursive=True); _tr = Path(_f[0]) if _f else None
assert _tr is not None, "trainer.py introuvable"
print(f"trainer.py : {_tr}")
_s = _tr.read_text()

MARK = "# _OPT_TRAIN_PATCHES_"
if MARK in _s:
    print("Patches d'optimisation déjà appliqués ✓")
else:
    # ── 1. TF32 + EMACallback (niveau module, après les imports) ─────────────
    _anchor = "from utils.utils_fm import build_model, validate_and_save_samples\n"
    _inject = _anchor + (
        "\n" + MARK + "\n"
        "import math as _math\n"
        "try:\n"
        "    torch.set_float32_matmul_precision('high')  # [OPT] TF32\n"
        "except Exception:\n"
        "    pass\n"
        "\n"
        "class EMACallback(pl.Callback):\n"
        "    \"\"\"EMA des poids. Les poids EMA sont ACTIFS pendant validation+checkpoint\n"
        "    (restaurés au début de l'epoch suivant), donc le .ckpt sauvegardé contient\n"
        "    les poids EMA. Compatible DDP (mêmes ops sur tous les rangs).\"\"\"\n"
        "    def __init__(self, decay=0.999):\n"
        "        super().__init__(); self.decay=float(decay); self.shadow={}; self.backup={}\n"
        "    def on_fit_start(self, trainer, pl_module):\n"
        "        if not self.shadow:\n"
        "            self.shadow={n: p.detach().clone().float()\n"
        "                         for n,p in pl_module.named_parameters() if p.requires_grad}\n"
        "    @torch.no_grad()\n"
        "    def on_train_batch_end(self, trainer, pl_module, *a, **k):\n"
        "        d=self.decay\n"
        "        for n,p in pl_module.named_parameters():\n"
        "            if p.requires_grad and n in self.shadow:\n"
        "                self.shadow[n].mul_(d).add_(p.detach().float(), alpha=1.0-d)\n"
        "    @torch.no_grad()\n"
        "    def on_train_epoch_start(self, trainer, pl_module):\n"
        "        # restaure les poids bruts si on avait basculé sur EMA à la validation\n"
        "        if self.backup:\n"
        "            for n,p in pl_module.named_parameters():\n"
        "                if n in self.backup: p.data.copy_(self.backup[n])\n"
        "            self.backup={}\n"
        "    @torch.no_grad()\n"
        "    def on_validation_epoch_start(self, trainer, pl_module):\n"
        "        if not self.shadow: return\n"
        "        self.backup={n: p.detach().clone() for n,p in pl_module.named_parameters()\n"
        "                     if n in self.shadow}\n"
        "        for n,p in pl_module.named_parameters():\n"
        "            if n in self.shadow: p.data.copy_(self.shadow[n].to(p.dtype))\n"
        "        # NB: on NE restaure PAS ici → checkpoint sauvegardé avec poids EMA.\n"
        "    def on_save_checkpoint(self, trainer, pl_module, checkpoint):\n"
        "        checkpoint['ema_shadow']={k:v.cpu() for k,v in self.shadow.items()}\n"
        "    def on_load_checkpoint(self, trainer, pl_module, checkpoint):\n"
        "        if 'ema_shadow' in checkpoint:\n"
        "            self.shadow={k:v.clone().float() for k,v in checkpoint['ema_shadow'].items()}\n"
    )
    assert _anchor in _s, "ancre imports introuvable"
    _s = _s.replace(_anchor, _inject, 1)

    # ── 2. configure_optimizers → AdamW + cosine + warmup ────────────────────
    _old_opt = (
        '    def configure_optimizers(self) -> optim.Optimizer:\n'
        '        lr = self.hparams["train_args"]["lr"]\n'
        '        return optim.Adam(self.model.parameters(), lr=lr)'
    )
    _new_opt = (
        '    def configure_optimizers(self):\n'
        '        ta = self.hparams["train_args"]\n'
        '        lr = ta["lr"]; wd = ta.get("weight_decay", 0.0)\n'
        '        opt = optim.AdamW(self.model.parameters(), lr=lr, weight_decay=wd)\n'
        '        if not ta.get("use_lr_scheduler", True):\n'
        '            return opt\n'
        '        try:\n'
        '            total = int(self.trainer.estimated_stepping_batches)\n'
        '        except Exception:\n'
        '            total = 0\n'
        '        if total <= 1:\n'
        '            return opt\n'
        '        warm = ta.get("warmup_steps", None)\n'
        '        if warm is None: warm = max(1, int(0.03 * total))\n'
        '        warm = min(int(warm), max(1, total - 1))\n'
        '        floor = float(ta.get("min_lr_ratio", 0.05))\n'
        '        def _fn(step):\n'
        '            if step < warm: return step / max(1, warm)\n'
        '            prog = (step - warm) / max(1, total - warm)\n'
        '            prog = min(1.0, max(0.0, prog))\n'
        '            return floor + (1.0 - floor) * 0.5 * (1.0 + _math.cos(_math.pi * prog))\n'
        '        sch = optim.lr_scheduler.LambdaLR(opt, _fn)\n'
        '        return {"optimizer": opt, "lr_scheduler": {"scheduler": sch, "interval": "step"}}'
    )
    assert _old_opt in _s, "configure_optimizers introuvable"
    _s = _s.replace(_old_opt, _new_opt, 1)

    # ── 3. torch.compile après construction du modèle ────────────────────────
    _old_m = "    model = FlowMatchingLightningModule(config)\n"
    _new_m = _old_m + (
        "    if tr.get('use_compile', False):\n"
        "        try:\n"
        "            model = torch.compile(model)\n"
        "            logger.info('[OPT] torch.compile activé.')\n"
        "        except Exception as _e:\n"
        "            logger.warning(f'[OPT] torch.compile indisponible ({_e}) — on continue sans.')\n"
    )
    assert _old_m in _s, "construction modèle introuvable"
    _s = _s.replace(_old_m, _new_m, 1)

    # ── 4. EMACallback ajouté aux callbacks ──────────────────────────────────
    _old_cb = "    cbs = [ckpt_cb, lr_cb]\n"
    _new_cb = _old_cb + (
        "    if tr.get('use_ema', False):\n"
        "        cbs.append(EMACallback(decay=float(tr.get('ema_decay', 0.999))))\n"
        "        logger.info(f\"[OPT] EMA activée (decay={tr.get('ema_decay', 0.999)}).\")\n"
    )
    assert _old_cb in _s, "cbs introuvable"
    _s = _s.replace(_old_cb, _new_cb, 1)

    # ── 5. limit_val_batches câblé dans le Trainer ───────────────────────────
    _old_t = "        num_sanity_val_steps=tr.get(\"num_sanity_val_steps\", 0),\n"
    _new_t = _old_t + "        limit_val_batches=tr.get(\"limit_val_batches\", 1.0),  # [OPT]\n"
    assert _old_t in _s, "Trainer num_sanity introuvable"
    _s = _s.replace(_old_t, _new_t, 1)

    # ── Vérification syntaxe puis écriture ───────────────────────────────────
    _tmp = tempfile.mktemp(suffix='.py')
    open(_tmp,'w').write(_s)
    try:
        py_compile.compile(_tmp, doraise=True)
        _tr.write_text(_s)
        print("trainer.py — patches d'optimisation appliqués, syntaxe OK ✓")
        print("  1. TF32  2. AdamW+cosine+warmup  3. torch.compile  "
              "4. limit_val_batches  5. EMA (poids sauvés = EMA)")
    except py_compile.PyCompileError as e:
        print("ERREUR syntaxe — patch NON écrit :", e)
    finally:
        os.unlink(_tmp)

trainer.py : MOTFM/trainer.py
trainer.py — patches d'optimisation appliqués, syntaxe OK ✓
  1. TF32  2. AdamW+cosine+warmup  3. torch.compile  4. limit_val_batches  5. EMA (poids sauvés = EMA)


In [23]:
# ── [LAZY v2] Patch trainer.py — Dataset paresseux (corrige split_val + val_data) ─
# inferer.py fait `from trainer import FlowMatchingDataModule` : le patch, inséré
# AVANT __main__, s'applique donc aussi à l'inférence de la §10.
#   v1 → v2 : (1) respecte data_args['split_train'/'split_val'] — sinon l'inférence
#   tournerait sur 'val' au lieu de 'test' ; (2) renseigne self.train_data/val_data
#   — sinon inferer.py lève "Failed to initialize validation data".
import glob, py_compile, tempfile, os, re
from pathlib import Path
_c=[Path('MOTFM/trainer.py'),Path('/kaggle/working/MOTFM/trainer.py'),Path('/root/MOTFM/trainer.py')]
_tr=next((p for p in _c if p.exists()), None)
if _tr is None:
    _f=glob.glob('**/trainer.py',recursive=True); _tr=Path(_f[0]) if _f else None
assert _tr, 'trainer.py introuvable'
print('trainer.py :', _tr)
_s=_tr.read_text()
_s=re.sub(r'\n# _LAZY_DATASET_PATCH_.*?# _LAZY_DATASET_PATCH_END_\n', '\n', _s, flags=re.S)
_block = '\n\n# _LAZY_DATASET_PATCH_  ── Dataset paresseux v2 (lecture .pt à la demande) ──────\n# v2 corrige : (1) respect de data_args[\'split_train\'/\'split_val\'] → l\'inférence\n# sur \'test\' fonctionne ; (2) self.train_data/self.val_data renseignés (contrat\n# attendu par trainer.py ET inferer.py, qui importe cette classe).\nimport pandas as _pd\nfrom pathlib import Path as _Path\nfrom torch.utils.data import Dataset as _LzDataset, DataLoader as _LzDataLoader\n\ndef _lz_minmax(t, eps=1e-6):\n    t = torch.nan_to_num(t.float(), nan=0.0, posinf=0.0, neginf=0.0)\n    mn = t.amin(); mx = t.amax()\n    return (t - mn) / (mx - mn).clamp_min(eps)\n\nclass LazyPtDataset(_LzDataset):\n    """Lit chaque .pt a la demande. rec[\'y\']=(1,H,W) cible -> \'images\' ;\n    rec[\'x\']=(3,H,W) conditionnement -> \'masks\'. Min-max par echantillon."""\n    def __init__(self, files, classes, tensors_dir, mask_conditioning,\n                 class_conditioning, num_classes=2, norm_scope="sample", eps=1e-6,\n                 modality_dropout=0.0, split="train"):\n        self.files = list(files)\n        self.classes = list(classes) if classes is not None else None\n        self.dir = _Path(tensors_dir)\n        self.mask_conditioning = bool(mask_conditioning)\n        self.class_conditioning = bool(class_conditioning)\n        self.num_classes = int(num_classes)\n        self.norm_scope = str(norm_scope); self.eps = float(eps)\n        self.modality_dropout = float(modality_dropout); self.split = str(split)\n    def __len__(self):\n        return len(self.files)\n    def _norm(self, t):\n        t = t.float()\n        if self.norm_scope == "sample_channel" and t.ndim >= 3:\n            flat = t.reshape(t.shape[0], -1)\n            shp = (-1,) + (1,) * (t.ndim - 1)\n            mn = flat.amin(1).reshape(shp); mx = flat.amax(1).reshape(shp)\n            return (t - mn) / (mx - mn).clamp_min(self.eps)\n        return _lz_minmax(t, self.eps)\n    def __getitem__(self, idx):\n        rec = torch.load(self.dir / self.files[idx], map_location="cpu", weights_only=True)\n        img = torch.as_tensor(rec["y"], dtype=torch.float32)\n        if img.ndim == 2: img = img.unsqueeze(0)\n        out = {"images": self._norm(img)}\n        if self.mask_conditioning:\n            m = torch.as_tensor(rec["x"], dtype=torch.float32)\n            if m.ndim == 2: m = m.unsqueeze(0)\n            m = self._norm(m)\n            if self.modality_dropout > 0.0 and self.split == "train" and m.shape[0] > 1:\n                import random as _rnd\n                if _rnd.random() < self.modality_dropout:\n                    _n = _rnd.choice([1, 1, 2])\n                    for _ch in _rnd.sample(range(m.shape[0]), min(_n, m.shape[0]-1)):\n                        m[_ch] = 0.0\n            out["masks"] = m\n        if self.class_conditioning and self.classes is not None:\n            c = int(self.classes[idx])\n            oh = torch.zeros(self.num_classes, dtype=torch.float32)\n            if 0 <= c < self.num_classes: oh[c] = 1.0\n            out["classes"] = oh\n        return out\n\ndef _lz_split_files(dc, split):\n    """Fichiers + classes pour un nom de split donne (ordre = slice_index.csv)."""\n    idx = _pd.read_csv(dc["slice_index_csv"])\n    spl = _pd.read_csv(dc["splits_csv"])\n    pid_col = "patient_id" if "patient_id" in spl.columns else spl.columns[0]\n    pids = set(spl.loc[spl["split"] == split, pid_col])\n    sub = idx[idx["pid"].isin(pids)]\n    files = sub["file"].tolist()\n    classes = sub["has_tumour"].astype(int).tolist() if "has_tumour" in sub.columns else None\n    return files, classes\n\n_lz_orig_setup = FlowMatchingDataModule.setup\n_lz_orig_train = FlowMatchingDataModule.train_dataloader\n_lz_orig_val   = FlowMatchingDataModule.val_dataloader\n\ndef _lz_make(self, dc, split_key, default):\n    """Construit (dataset lazy, meta-dict compatible trainer/inferer)."""\n    split = dc.get(split_key, default)\n    files, classes = _lz_split_files(dc, split)\n    ncls = int(self.config.get("model_args", {}).get("cross_attention_dim", 2) or 2)\n    ds = LazyPtDataset(files, classes, dc["tensors_dir"], self.mask_conditioning,\n                       self.class_conditioning, num_classes=ncls,\n                       norm_scope=dc.get("norm_scope", "sample"),\n                       modality_dropout=dc.get("modality_dropout", 0.0),\n                       split=split)\n    # meta : tenseurs vides (0 octet) juste pour exposer .shape[0] + class_map\n    meta = {"images": torch.empty((len(ds), 0)),\n            "class_map": {i: i for i in range(ncls)}}\n    if self.mask_conditioning:  meta["masks"]   = torch.empty((len(ds), 0))\n    if self.class_conditioning: meta["classes"] = torch.empty((len(ds), 0))\n    logger.info(f"[LAZY] split \'{split}\' -> {len(ds)} echantillons (lecture .pt a la demande)")\n    return ds, meta\n\ndef _lz_setup(self, stage=None):\n    dc = self.config["data_args"]\n    if not dc.get("lazy_loading"):\n        return _lz_orig_setup(self, stage)\n    if stage in (None, "fit"):\n        self._lz_train, self.train_data = _lz_make(self, dc, "split_train", "train")\n        self._lz_val,   self.val_data   = _lz_make(self, dc, "split_val",   "val")\n    elif stage == "validate":\n        self._lz_val,   self.val_data   = _lz_make(self, dc, "split_val",   "val")\n\ndef _lz_make_dl(self, ds, shuffle):\n    tr = self.config["train_args"]\n    nw = int(tr.get("num_workers", 4))\n    kw = dict(batch_size=tr["batch_size"], shuffle=shuffle, num_workers=nw,\n              pin_memory=tr.get("pin_memory", torch.cuda.is_available()),\n              persistent_workers=(nw > 0),\n              drop_last=bool(tr.get("drop_last", False)) and shuffle)\n    if nw > 0: kw["prefetch_factor"] = int(tr.get("prefetch_factor", 4))\n    return _LzDataLoader(ds, **kw)\n\ndef _lz_train_dl(self):\n    if not self.config["data_args"].get("lazy_loading"): return _lz_orig_train(self)\n    return _lz_make_dl(self, self._lz_train, shuffle=True)\n\ndef _lz_val_dl(self):\n    if not self.config["data_args"].get("lazy_loading"): return _lz_orig_val(self)\n    return _lz_make_dl(self, self._lz_val, shuffle=False)\n\nFlowMatchingDataModule.setup = _lz_setup\nFlowMatchingDataModule.train_dataloader = _lz_train_dl\nFlowMatchingDataModule.val_dataloader = _lz_val_dl\n# _LAZY_DATASET_PATCH_END_\n'
_anchor='if __name__ == "__main__":'
assert _anchor in _s, 'ancre __main__ introuvable'
_s=_s.replace(_anchor, _block + '\n' + _anchor, 1)
_tmp=tempfile.mktemp(suffix='.py'); open(_tmp,'w').write(_s)
try:
    py_compile.compile(_tmp, doraise=True); _tr.write_text(_s)
    print('Patch LAZY v2 appliqué, syntaxe OK ✓  (split_val respecté + val_data renseigné)')
except py_compile.PyCompileError as _e:
    print('ERREUR syntaxe — patch NON écrit :', _e)
finally:
    os.unlink(_tmp)

trainer.py : MOTFM/trainer.py
Patch LAZY v2 appliqué, syntaxe OK ✓  (split_val respecté + val_data renseigné)


In [24]:
import os
import kagglehub
from pathlib import Path

# Configuration des identifiants
os.environ["KAGGLE_USERNAME"] = "donvins"
os.environ["KAGGLE_KEY"] = "c70a5d3ab3b4cf94694ec04a2f7b6d06"

try:
    print(f"Tentative de téléchargement depuis le dataset public 'donvins/outputs'...")
    # Téléchargement via kagglehub
    path = kagglehub.dataset_download("donvins/outputs")
    print(f"\nFichiers téléchargés dans : {path}")

    # Inventaire des checkpoints
    ckpt_files = list(Path(path).rglob("*.ckpt"))
    if ckpt_files:
        print(f"\n{len(ckpt_files)} checkpoint(s) trouvé(s) :")
        # Tri par date de modification (plus récent en premier)
        for c in sorted(ckpt_files, key=lambda x: x.stat().st_mtime, reverse=True):
            size_mb = c.stat().st_size / (1024 * 1024)
            print(f" - {c.name} ({size_mb:.1f} Mo) | Chemin: {c}")

        # Stockage du chemin du plus récent pour la suite
        LATEST_KAGGLE_CKPT = str(sorted(ckpt_files, key=lambda x: x.stat().st_mtime, reverse=True)[0])
        print(f"\nCheckpoint le plus récent mémorisé : {LATEST_KAGGLE_CKPT}")
    else:
        print("\nAttention : Aucun fichier .ckpt détecté dans le dossier téléchargé.")

except Exception as e:
    print(f"\nErreur lors du téléchargement : {e}")

Tentative de téléchargement depuis le dataset public 'donvins/outputs'...

Erreur lors du téléchargement : POST failed with: {"errors":["New Datasets cannot be attached in non-interactive sessions. Found no versions attached for Dataset [donvins/outputs]."],"error":{"code":9},"wasSuccessful":false}


In [25]:
# ── Détecte le dernier checkpoint et l'injecte dans le YAML ─────────────────
import glob, yaml
from pathlib import Path

ckpt_root = Path(CKPT_DIR) / 'motfm_config'
candidates = sorted(
    glob.glob(str(ckpt_root / '**' / '*.ckpt'), recursive=True),
    key=lambda p: Path(p).stat().st_mtime
)

if candidates:
    last_ckpt = candidates[-1]
    print(f"Checkpoint trouvé : {last_ckpt}")
    # Injecte dans le YAML
    with open(config_path) as f: cfg = yaml.safe_load(f)
    cfg.setdefault('train_args', {})['ckpt_path'] = last_ckpt
    with open(config_path, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)
    print(f"ckpt_path injecté dans {config_path} ✓")
else:
    print("Aucun checkpoint trouvé — entraînement depuis zéro")
    # Retire ckpt_path si présent
    with open(config_path) as f: cfg = yaml.safe_load(f)
    cfg.get('train_args', {}).pop('ckpt_path', None)
    with open(config_path, 'w') as f: yaml.safe_dump(cfg, f, sort_keys=False)


Aucun checkpoint trouvé — entraînement depuis zéro


## 9. Training (trainer.py)

In [26]:
# ⚠️  ENTRAÎNEMENT DÉJÀ TERMINÉ (50/50 époques, code de sortie 0, 8,69 h).
# Cellule DÉSACTIVÉE par défaut : la relancer ne ferait rien (Lightning s'arrête
# aussitôt, max_epochs atteint). Pour ré-entraîner : RUN_TRAINING=True ET
# augmenter CFG.max_epochs dans la cellule Config.
RUN_TRAINING = True   # réentraînement AVEC modality dropout
if not RUN_TRAINING:
    print('[Skip] Entraînement déjà terminé — passe à la restauration du checkpoint.')
else:
    # ── Full training (multi-GPU DDP) ───────────────────────────────────────────
    # Multi-GPU réglé dans la cellule YAML (§7). Ici on lance trainer.py en
    # sous-processus EN CAPTURANT LE CODE DE SORTIE → un OOM (process tué) devient
    # visible (code 137 / -9) au lieu d'un « terminé » silencieux.
    #
    # Anti-OOM RAM : env ci-dessous + (YAML §7) num_workers=0 / pin_memory=False en DDP.
    # Anti-OOM VRAM : expandable_segments + charge 4/GPU inchangée.

    import time, os, subprocess

    env = os.environ.copy()
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    env['PYTORCH_ALLOC_CONF']      = 'expandable_segments:True'
    env['TOKENIZERS_PARALLELISM']  = 'false'
    env.setdefault('OMP_NUM_THREADS', '1')

    # (Optionnel mais utile) libère la copie du dataset détenue par le kernel
    # notebook avant de lancer 2 processus DDP — récupère quelques Go de RAM.
    for _v in ('Images', 'Masks', 'data_dict', '_df', 'df_slices'):
        if _v in globals():
            try: del globals()[_v]
            except Exception: pass
    import gc; gc.collect()

    t0 = time.time()
    proc = subprocess.run(
        ['python', 'trainer.py', '--config_path', str(config_path)],
        cwd='MOTFM', env=env,
    )
    rc = proc.returncode
    dt = (time.time() - t0) / 3600
    print(f"\n[trainer.py] code de sortie = {rc}  |  durée = {dt:.2f} h")

    if rc == 0:
        print("Entraînement terminé ✓")
    elif rc in (137, -9, 247):
        print("✗ OOM : un processus a été TUÉ (SIGKILL). La RAM hôte (30 Go) a été "
              "dépassée par les 2 copies DDP du dataset.\n"
              "  → Dans la cellule §7, garde DDP_NUM_WORKERS=0 (déjà le cas) et, si "
              "ça persiste :\n"
              "    • baisse CFG.dataset_fraction (ex. 0.1) puis régénère le PKL (§8) ;\n"
              "    • ou reste sur 1 seul GPU (devices=1) qui tenait déjà en RAM.")
    elif rc in (-6, 134):
        print("✗ Abort (souvent NCCL/CUDA). Vérifie les logs ci-dessus.")
    else:
        print(f"✗ Échec (code {rc}). Voir la traceback ci-dessus.")


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
/usr/local/lib/python3.12/dist-packages/generative/networks/layers/vector_quantizer.py:86: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
/usr/local/lib/python3.12/dist-packages/generative/networks/layers/vector_quantizer.py:124: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @torch.cuda.amp.autocast(enabled=False)
[INFO] Loaded config file: /kaggle/working/outputs/run_motfm_full_100pct/motfm_config.yaml
[INFO] Loaded training config: /kaggle/working/outputs/run_motfm_full_100pct/motfm_config.yaml
[INFO] Run name: motfm_config
[INFO] Checkpoint root directory: /kaggle/working/outputs/run_motfm_fu

Epoch 0:   0%|          | 0/598 [00:00<?, ?it/s]

[rank0]:W0726 23:35:09.532000 187 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode
[rank1]:W0726 23:35:09.552000 196 torch/_inductor/utils.py:1679] [0/0_1] Not enough SMs to use max_autotune_gemm mode
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reductio

Epoch 0: 100%|█████████▉| 597/598 [19:18<00:01,  0.52it/s, v_num=0, train/loss_step=0.0236]

/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is disabled on the fly since Inductor decides to
split the reduction. Cut an issue to PyTorch if this is an
important use case and you want to speed it up with online
softmax.

  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:7627: UserWarning: 
Online softmax is dis

...


Epoch 49: 100%|██████████| 598/598 [09:46<00:00,  1.02it/s, v_num=0, train/loss_step=0.00956, train/loss_epoch=0.013, val/loss=0.0161]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 21/21 [00:05<00:00,  4.17it/s]

[INFO] Running validation sample export for epoch 49.
[INFO] Saving validation samples: epoch=49, max_samples=8, output_dir=/kaggle/working/outputs/run_motfm_full_100pct/motfm_checkpoints/motfm_config/version_0/val_samples_epoch_49.


Validating:   0%|          | 0/170 [00:09<?, ?it/s]
[INFO] Validation samples saved in: /kaggle/working/outputs/run_motfm_full_100pct/motfm_checkpoints/motfm_config/version_0/val_samples_epoch_49



Epoch 49: 100%|██████████| 598/598 [10:05<00:00,  0.99it/s, v_num=0, train/loss_step=0.00956, train/loss_epoch=0.0129, val/loss=0.0193]

`Trainer.fit` stopped: `max_epochs=50` reached.




[trainer.py] code de sortie = 0  |  durée = 8.69 h
Entraînement terminé ✓


## Modality-dropout evaluation (test-time) + prediction export
Tests all 7 modality combinations on the retrained model and saves predictions to `modality_dropout_eval/` for the supervisor.

In [27]:
# ── Modality-dropout evaluation at TEST time + SAVE predictions ──────────────
# Tests all 7 available-modality combinations on the (re)trained model and
# exports predictions to a folder (per supervisor's request). Runs on BraTS-MEN
# test slices so the ONLY variable is which modalities are present.
import torch, numpy as np, glob, json
from pathlib import Path
from PIL import Image
from skimage.metrics import structural_similarity as sk_ssim

OUT = Path(CFG.out_dir) / "modality_dropout_eval"
(OUT / "png").mkdir(parents=True, exist_ok=True)

# Collect a handful of BraTS test tensors (adjust glob if needed)
cand = sorted(glob.glob("/kaggle/input/**/*BraTS-MEN*_z*.pt", recursive=True)) \
    or sorted(glob.glob(str(Path(CFG.tensors_dir) / "*BraTS-MEN*_z*.pt")))
N = 16
cand = cand[:N]
assert cand, "No BraTS .pt found — point the glob at your tensors dir."

def _mm(t): t = t.float(); return (t - t.amin()) / (t.amax() - t.amin()).clamp_min(1e-6)
X = torch.stack([_mm(torch.as_tensor(torch.load(f, weights_only=True)["x"]).float()) for f in cand])
Y = torch.stack([_mm(torch.as_tensor(torch.load(f, weights_only=True)["y"]).float()) for f in cand])

names = ["t1n", "t2w", "t2f"]
combos = {"all_3": [0,1,2],
          "drop_t1n": [1,2], "drop_t2w": [0,2], "drop_t2f": [0,1],
          "only_t1n": [0], "only_t2w": [1], "only_t2f": [2]}
mc = _cfg.get('model_args', {})
sc = {"method": "midpoint", "time_points": CFG.nfe if hasattr(CFG,'nfe') else 100,
      "step_size": 1.0 / (CFG.nfe if hasattr(CFG,'nfe') else 100)}
def n01(a): a = np.asarray(a, np.float32); return (a - a.min()) / (np.ptp(a) + 1e-8)

results = {}
for name, keep in combos.items():
    Xd = torch.zeros_like(X)
    for c in keep: Xd[:, c] = X[:, c]
    batch = {"images": Y, "masks": Xd,
             "classes": torch.zeros(len(X), 2).scatter_(1, torch.ones(len(X),1).long(), 1.0)}
    torch.manual_seed(0)
    with torch.no_grad():
        out = sample_batch(model_module.model, sc, batch, torch.device(DEVICE),
                           class_conditioning=bool(mc.get('with_conditioning', False)),
                           mask_conditioning=bool(mc.get('mask_conditioning', False)))
    p = out.detach().float().cpu().numpy()[:, 0]; g = Y.numpy()[:, 0]
    ss = [float(sk_ssim(n01(p[i]), n01(g[i]), data_range=1.0)) for i in range(len(p))]
    results[name] = {"ssim_mean": float(np.mean(ss)), "ssim_std": float(np.std(ss)),
                     "kept": [names[c] for c in keep]}
    # save all predictions of this combo
    np.savez_compressed(OUT / f"{name}.npz", pred=p, gt=g, kept=keep)
    for i in range(min(4, len(p))):        # a few PNGs per combo
        Image.fromarray((n01(p[i])*255).astype(np.uint8)).save(OUT/"png"/f"{name}_s{i}_pred.png")
    print(f"{name:10s} keep={'+'.join(names[c] for c in keep):12s} → SSIM {np.mean(ss):.4f} ± {np.std(ss):.4f}")

# reference target/input PNGs
for i in range(min(4, len(Y))):
    Image.fromarray((n01(g[i])*255).astype(np.uint8)).save(OUT/"png"/f"_target_t1ce_s{i}.png")
    Image.fromarray((n01(X.numpy()[i,0])*255).astype(np.uint8)).save(OUT/"png"/f"_input_t1n_s{i}.png")

json.dump(results, open(OUT / "summary.json", "w"), indent=2)
print(f"\nPredictions + PNGs + summary.json saved to: {OUT}")
print("→ Save Version (Kaggle) OR push this folder to Hugging Face to share with supervisor.")
print("\nComparison vs pre-dropout model (t1n-only was 0.942, drop_t1n collapsed):")
print("  If drop_t1n is now HIGH, the model learned to use t2w/t2f — dropout worked.")

NameError: name '_cfg' is not defined

## 10. Full Test-Set Inference — NFE=100, all 2,573 slices

**Per Samia:** minimum 100 integration steps (FAST-DDPM style). Evaluating the full test set (not a 120-slice subset) is required for a statistically valid EVarΔ.

In [ ]:
# ── Full test-set inference: NFE=100, all slices ────────────────────────────
import os, sys, glob, yaml, pickle
from pathlib import Path
import numpy as np, torch
from skimage.metrics import structural_similarity as sk_ssim
from tqdm import tqdm

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_RANGE = 1.0
CFG.metric_data_range = DATA_RANGE

# ── Samia recommendation: NFE=100 minimum (was 10) ──────────────────────────
NFE   = 100
N_MAX = None          # None → evaluate ALL test slices (2,573)

MOTFM_DIR = Path('MOTFM').resolve()
assert MOTFM_DIR.exists(), "MOTFM directory missing — rerun the clone cell."
if str(MOTFM_DIR) not in sys.path:
    sys.path.insert(0, str(MOTFM_DIR))
from trainer import FlowMatchingLightningModule, FlowMatchingDataModule
from utils.utils_fm import sample_batch

# 1) Checkpoint (+ fall back to Kaggle input if local is missing)
# Fixed: correctly terminated string literal in glob pattern
_ckpts = sorted(glob.glob(str(Path(CKPT_DIR) / '**' / '*.ckpt'), recursive=True),
                key=lambda p: Path(p).stat().st_mtime)

if not _ckpts and 'LATEST_KAGGLE_CKPT' in globals():
    print(f"Local checkpoint missing. Falling back to Kaggle input: {LATEST_KAGGLE_CKPT}")
    BEST_CKPT = LATEST_KAGGLE_CKPT
else:
    assert _ckpts, f"No .ckpt found in {CKPT_DIR}. Please ensure training finished or checkpoints are mounted."
    BEST_CKPT = _ckpts[-1]

_ck = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
if any(k.startswith("_orig_mod.") for k in _ck.get("state_dict", {})):
    _ck["state_dict"] = {k.replace("_orig_mod.", "", 1): v for k, v in _ck["state_dict"].items()}
    # Save a local fixed copy since /kaggle/input is read-only
    FIXED_CKPT = Path(CFG.persist_base)/"fixed_inference.ckpt"
    torch.save(_ck, FIXED_CKPT)
    BEST_CKPT = str(FIXED_CKPT)
    print("torch.compile prefix stripped and saved to local disk ✓")
print("Using Checkpoint:", BEST_CKPT)

# 2) Config → test split
with open(config_path) as f: _cfg = yaml.safe_load(f)
_cfg.setdefault('data_args', {})['split_val'] = 'test'
_cfg['train_args']['batch_size']  = 8
_cfg['train_args']['num_workers'] = 2

model_module = FlowMatchingLightningModule.load_from_checkpoint(
    BEST_CKPT, config=_cfg, map_location=DEVICE)
model_module.to(DEVICE).eval()

dm = FlowMatchingDataModule(_cfg)
dm.setup(stage="validate")
loader = dm.val_dataloader()          # shuffle=False → order matches test_idx
print("test:", len(loader.dataset), "samples")

# 3) Generation at NFE=100
mc = _cfg.get('model_args', {})
solver_config = {"method": "midpoint", "time_points": NFE, "step_size": 1.0/NFE}
preds, gts, masks = [], [], []
with torch.no_grad():
    for batch in tqdm(loader, desc=f"MOTFM inference (NFE={NFE})"):
        out = sample_batch(model_module.model, solver_config, batch, torch.device(DEVICE),
                           class_conditioning=bool(mc.get('with_conditioning', False)),
                           mask_conditioning=bool(mc.get('mask_conditioning', False)))
        preds.extend(out.detach().float().cpu().numpy()[:, 0])
        gts.extend(batch["images"].numpy()[:, 0])
        masks.extend(batch["masks"].numpy())
        if N_MAX is not None and len(preds) >= N_MAX: break
if N_MAX is not None:
    preds, gts, masks = preds[:N_MAX], gts[:N_MAX], masks[:N_MAX]
print(f"{len(preds)} predictions generated at NFE={NFE}")

# 4) Save pickle + build aligned test_list
CFG.out_dir.mkdir(parents=True, exist_ok=True)
PKL = CFG.out_dir / f"samples_nfe{NFE}.pkl"
with open(PKL, "wb") as f:
    pickle.dump({"test": [{"image": p[None, ...]} for p in preds]}, f)
print("saved:", PKL)

rows = list(test_idx.itertuples(index=False))[:len(preds)]
test_list = [{
    "metadata": {"pid": r.pid, "z": int(r.z), "file": r.file},
    "class":    int(getattr(r, "has_tumour", 0)),
    "image":    gts[i][None, ...],
    "mask":     masks[i],
} for i, r in enumerate(rows)]

opt_nfe = NFE
def _n01(a):
    a = np.asarray(a, np.float32); lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo + 1e-8)
ss = [float(sk_ssim(_n01(preds[i]), _n01(gts[i]), data_range=DATA_RANGE)) for i in range(len(preds))]
print(f"NFE={NFE} — SSIM {np.mean(ss):.4f} ± {np.std(ss):.4f} over {len(ss)} slices")

## 11. 3D TIFF Volume Export (Samia point 1)

Assemble per-slice predictions into 3D volumes for **≥20 patients**. For each patient, both the GT T1CE and the MOTFM-predicted T1CE are saved as 3D TIFF stacks, enabling stackwise comparison.

```
predictions/
  BraTS-MEN-XXXXX-000/
    t1ce_gt.tif          # ground-truth T1CE volume (Z, H, W)
    t1ce_motfm_pred.tif  # MOTFM-predicted T1CE volume (Z, H, W)
```

In [ ]:
# ── Assemble per-slice predictions into 3D TIFF volumes ─────────────────────
import numpy as np
from pathlib import Path
from collections import defaultdict
try:
    import tifffile
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tifffile"])
    import tifffile

PRED_ROOT = CFG.out_dir / "predictions_3d"
PRED_ROOT.mkdir(parents=True, exist_ok=True)

MIN_PATIENTS = 20        # Samia: at least 20 volumes

# Group slice indices by patient, ordered by axial position z
by_pid = defaultdict(list)
for i in range(len(test_list)):
    by_pid[test_list[i]["metadata"]["pid"]].append(i)

def _n01(a):
    a = np.asarray(a, np.float32); lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo + 1e-8)

# Prioritise patients with the most slices (most complete volumes)
pids_sorted = sorted(by_pid.keys(), key=lambda p: -len(by_pid[p]))
exported = 0

for pid in pids_sorted:
    sl = sorted(by_pid[pid], key=lambda i: test_list[i]["metadata"]["z"])
    if len(sl) < 5:      # skip patients with too few slices for a meaningful volume
        continue

    gt_stack   = np.stack([_n01(np.asarray(test_list[i]["image"])[0]) for i in sl], axis=0)
    pred_stack = np.stack([_n01(preds[i]) for i in sl], axis=0)

    pdir = PRED_ROOT / pid
    pdir.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(pdir / "t1ce_gt.tif",         (gt_stack   * 255).astype(np.uint8))
    tifffile.imwrite(pdir / "t1ce_motfm_pred.tif", (pred_stack * 255).astype(np.uint8))

    exported += 1
    print(f"  [{exported:2d}] {pid}: {len(sl)} slices → t1ce_gt.tif + t1ce_motfm_pred.tif")
    if exported >= max(MIN_PATIENTS, 20) and exported >= MIN_PATIENTS:
        # keep exporting until we have at least MIN_PATIENTS; stop once we have enough
        if exported >= MIN_PATIENTS:
            break

print(f"\n✓ Exported {exported} patient volumes → {PRED_ROOT}")
print(f"  Structure: predictions_3d/<pid>/{{t1ce_gt.tif, t1ce_motfm_pred.tif}}")

In [ ]:
# ── Stackwise comparison figure for a few exported volumes ──────────────────
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

vol_dirs = sorted([d for d in (CFG.out_dir/"predictions_3d").iterdir() if d.is_dir()])[:3]

fig, axes = plt.subplots(len(vol_dirs), 6, figsize=(15, 2.6*len(vol_dirs)))
if len(vol_dirs) == 1: axes = axes[None, :]

for r, vd in enumerate(vol_dirs):
    gt   = tifffile.imread(vd/"t1ce_gt.tif")
    pred = tifffile.imread(vd/"t1ce_motfm_pred.tif")
    Z = gt.shape[0]
    zsel = np.linspace(Z//5, 4*Z//5, 3).astype(int)   # 3 representative slices
    for c, z in enumerate(zsel):
        axes[r, c*2].imshow(gt[z],   cmap='gray'); axes[r, c*2].set_title(f"GT z={z}", fontsize=8)
        axes[r, c*2+1].imshow(pred[z], cmap='gray'); axes[r, c*2+1].set_title(f"Pred z={z}", fontsize=8)
        axes[r, c*2].axis('off'); axes[r, c*2+1].axis('off')
    axes[r, 0].set_ylabel(vd.name, fontsize=8)

plt.suptitle("3D Volume stackwise comparison — GT vs MOTFM predicted T1CE", fontsize=11)
plt.tight_layout()
plt.savefig(CFG.out_dir/"volume_stackwise_comparison.png", dpi=100, bbox_inches='tight')
plt.show()
print("→ volume_stackwise_comparison.png saved")

## 12. Full Metrics — EVarΔ / Wasserstein / Multimodality

Now computed on the **full test set** (all tumour patients, not n=1).

In [ ]:
# ═══ Full-test-set metrics — domain [0,1] ═══════════════════════════════════
import pickle, json, numpy as np
from collections import defaultdict
from scipy.stats import wasserstein_distance
from skimage.metrics import structural_similarity as sk_ssim, peak_signal_noise_ratio as sk_psnr

NFE_EVAL = int(opt_nfe) if 'opt_nfe' in dir() else 100
THRESH   = 0.05

def _n01(a):
    a = np.asarray(a, np.float32); lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo + 1e-8)
def _gt_image(s):
    g = np.asarray(s['image']); return g[0] if g.ndim == 3 else g

assert 'test_list' in dir(), "Run §10 first (it builds test_list)."

# Predictions already in memory from §10 (variable `preds`); reload if needed
if 'preds' not in dir():
    with open(CFG.out_dir/f"samples_nfe{NFE_EVAL}.pkl","rb") as f: gen = pickle.load(f)
    key   = 'test' if 'test' in gen else next(iter(gen))
    preds = [np.asarray(s['image'], np.float32) for s in gen[key]]
    preds = [p[0] if p.ndim == 3 else p for p in preds]

M = min(len(preds), len(test_list))

# Normalise
pred01, gt01, t1n01, meta = [], [], [], []
for i in range(M):
    s = test_list[i]
    pred01.append(_n01(preds[i]))
    gt01.append(_n01(_gt_image(s)))
    t1n01.append(_n01(np.asarray(s['mask'])[0]))
    meta.append((s['metadata']['pid'], s['metadata']['z'], bool(s['class'])))

# SSIM / PSNR
ssim_all = [float(sk_ssim(pred01[i], gt01[i], data_range=1.0)) for i in range(M)]
psnr_all = [float(sk_psnr(gt01[i], pred01[i], data_range=1.0)) for i in range(M)]
tum      = [i for i in range(M) if meta[i][2]]
heal     = [i for i in range(M) if not meta[i][2]]
ssim_tum  = float(np.mean([ssim_all[i] for i in tum]))  if tum  else None
ssim_heal = float(np.mean([ssim_all[i] for i in heal])) if heal else None

# EVarΔ — now over ALL tumour patients (fixes n=1)
by_pid = defaultdict(list)
for i in range(M):
    if meta[i][2]: by_pid[meta[i][0]].append(i)
evar_vals = []
for pid, sl in by_pid.items():
    sl = sorted(sl, key=lambda i: meta[i][1])
    dp, dg = [], []
    for i in sl:
        enh = (gt01[i] - t1n01[i]) > THRESH
        if enh.sum() < 10: continue
        dp.append(float((pred01[i] - t1n01[i])[enh].mean()))
        dg.append(float((gt01[i]  - t1n01[i])[enh].mean()))
    if len(dg) >= 2:
        vg = float(np.var(dg, ddof=1))
        if vg > 1e-8: evar_vals.append(float(np.var(dp, ddof=1)) / vg)
evar_mean = float(np.mean(evar_vals)) if evar_vals else None
evar_std  = float(np.std(evar_vals))  if evar_vals else None

# Wasserstein-1
ap, ag = [], []
for i in range(M):
    if not meta[i][2]: continue
    enh = (gt01[i] - t1n01[i]) > THRESH
    if enh.sum() < 5: continue
    ap.extend((pred01[i] - t1n01[i])[enh].tolist())
    ag.extend((gt01[i]  - t1n01[i])[enh].tolist())
if ag:
    rng = np.random.default_rng(42); N = min(len(ag), 100_000)
    d_W = float(wasserstein_distance(rng.choice(ag, N, replace=False),
                                     rng.choice(ap, N, replace=False)))
else:
    d_W = None

# Multimodality
try:
    train_pids = set(splits_df.loc[splits_df['split']=='train','patient_id'])
    train_m = idx[idx['pid'].isin(train_pids) & (idx['has_tumour']==True)]
    multi = prove_multimodality(train_m, CFG.tensors_dir, n_pairs=200)
    multi_L1 = multi.get('matched_L1_mean')
except Exception as e:
    multi_L1 = None; print("multimodality skipped:", e)

print(f"════ MOTFM — NFE={NFE_EVAL} — {M} slices — {len(by_pid)} tumour patients ════")
print(f"SSIM  (all)      : {np.mean(ssim_all):.4f} ± {np.std(ssim_all):.4f}")
print(f"SSIM  (tumour)   : {ssim_tum:.4f}" if ssim_tum else "SSIM (tumour): n/a")
print(f"SSIM  (healthy)  : {ssim_heal:.4f}" if ssim_heal else "SSIM (healthy): n/a")
print(f"PSNR  (all)      : {np.mean(psnr_all):.2f} dB")
print(f"EVarΔ            : {evar_mean:.4f} ± {evar_std:.4f}  (n={len(evar_vals)} patients)" if evar_mean else "EVarΔ: n/a")
print(f"Wasserstein d_W  : {d_W:.4f}  ({len(ag):,} voxels)" if d_W else "d_W: n/a")
print(f"Multimodality L1 : {multi_L1:.4f}" if multi_L1 else "Multimodality: n/a")

summary = {"model":"MOTFM","nfe":NFE_EVAL,"n_slices":M,"n_tumour_patients":len(by_pid),
           "ssim":float(np.mean(ssim_all)),"ssim_tumour":ssim_tum,"ssim_healthy":ssim_heal,
           "psnr":float(np.mean(psnr_all)),"evar_mean":evar_mean,"evar_std":evar_std,
           "evar_n":len(evar_vals),"d_W":d_W,"multi_L1_mean":multi_L1}
with open(CFG.out_dir/"metrics_summary.json","w") as f: json.dump(summary,f,indent=2,default=str)
print("\n→ metrics_summary.json saved")

## 13. Clean Metrics Table — Yazdani et al. Style (Samia point 3)

Only models with valid (non-anomalous) results: Identity · U-Net · U-Net+LPIPS · MOTFM. DDPM excluded until its normalisation bug is fixed.

In [ ]:
# ── Clean comparison table (Yazdani et al. MICCAI 2025 Table 1 style) ───────
import pandas as pd

# MOTFM row from the current run; others from prior completed evaluations
rows = [
    # model,             SSIM_all, SSIM_tum, SSIM_heal, PSNR,  MAE,   d_W
    ("Identity (T1n)",   0.765,    0.721,    0.789,     17.83, 0.562, None),
    ("U-Net",            0.873,    0.872,    0.874,     25.97, 0.185, None),
    ("U-Net + LPIPS",    0.872,    0.856,    0.879,     26.32, 0.174, None),
    ("MOTFM (ours)",     round(float(np.mean(ssim_all)),3),
                         round(ssim_tum,3) if ssim_tum else None,
                         round(ssim_heal,3) if ssim_heal else None,
                         round(float(np.mean(psnr_all)),2), None,
                         round(d_W,3) if d_W else None),
]
df = pd.DataFrame(rows, columns=["Model","SSIM (all)","SSIM (tum)","SSIM (heal)","PSNR (dB)","MAE","d_W"])
print(df.to_string(index=False))
df.to_csv(CFG.out_dir/"clean_metrics_table.csv", index=False)
print("\n→ clean_metrics_table.csv saved")

# Styled display
try:
    from IPython.display import display
    display(df.style.format(na_rep="—", precision=3)
              .set_caption("synT1CE — Clean Metrics Table (Yazdani et al. style)")
              .highlight_max(subset=["SSIM (all)","SSIM (tum)","SSIM (heal)","PSNR (dB)"], color="#d4f0e0")
              .highlight_min(subset=["MAE","d_W"], color="#d4f0e0"))
except Exception:
    pass

## 15. Package & Download Results

In [ ]:
# ── Zip everything (3D TIFFs, metrics, tables) for download / upload ────────
import zipfile
from pathlib import Path

zp = CFG.persist_base / "synT1CE_motfm_final_results.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in ["metrics_summary.json", "clean_metrics_table.csv",
                    "metrics_peds_ood.json", "volume_stackwise_comparison.png",
                    "diag_triplets.png"]:
        p = CFG.out_dir / pattern
        if p.exists(): zf.write(p, pattern); print(f"  + {pattern}")
    # 3D TIFF volumes (BraTS-MEN)
    for tif in (CFG.out_dir/"predictions_3d").rglob("*.tif"):
        zf.write(tif, f"predictions_3d/{tif.parent.name}/{tif.name}")
    # 3D TIFF volumes (BraTS-PEDs OOD)
    peds_dir = CFG.out_dir/"predictions_peds_3d"
    if peds_dir.exists():
        for tif in peds_dir.rglob("*.tif"):
            zf.write(tif, f"predictions_peds_3d/{tif.parent.name}/{tif.name}")

print(f"\n✓ ZIP: {zp}  ({zp.stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files; files.download(str(zp))
except Exception:
    print(f"File available at: {zp}")